In [1]:
import os
import numpy as np
import yaml
import matplotlib.pyplot as plt
import healpy as hp
import heracles
import heracles.dices as dices
from heracles.io import read
from astropy.io import fits

font = {'size'   : 16}
import matplotlib
matplotlib.rc('font', **font)
matplotlib.rc('xtick', labelsize=12) 
matplotlib.rc('ytick', labelsize=12) 

# Comparison

## Masks

In [ ]:
config_path = "scripts/sims_config.yaml"
with open(config_path, 'r') as f:
    config = yaml.safe_load(f)
n = config['nsims']
nside = config['nside']
lmax = config['lmax']
lmax_mask = config.get('lmax_mask', lmax)  # Default to lmax if not specified
mode = config['mode']  # "lognormal" or "gaussian"
mask_type = config['mask_type']  # Default to 'dr1' if not specified
binned = False #True
apply_mask = True
nbins = 3

output_path = f"{mode}_dices/"
output_path = "./masked_"+output_path

nlbins = config.get('nlbins', 20)  # Default to 20 if not specified
l = np.arange(lmax+1)
l_mask = np.arange(lmax_mask+1)
ledges = np.logspace(np.log10(10), np.log10(lmax), nlbins + 1)
lgrid = (ledges[1:] + ledges[:-1]) / 2

: 

: 

In [ ]:
mask_patch = hp.read_map("./patch/mask.fits")
mask_dr1 = hp.read_map("./dr1/mask.fits")
mask_rr2 = hp.read_map("./rr2/mask.fits")

: 

: 

In [ ]:
fsky_rr2=np.sum(mask_rr2)/len(mask_rr2)
fsky_dr1=np.sum(mask_dr1)/len(mask_dr1)
fsky_patch=np.sum(mask_patch)/len(mask_patch)
print(fsky_rr2, fsky_patch, fsky_dr1)

: 

: 

In [ ]:
fig, (ax1, ax2, ax3) = plt.subplots(ncols=3, figsize=(15, 10))

plt.axes(ax1)
hp.mollview(mask_rr2, hold=True, title="RR2 mask")
plt.axes(ax2)
hp.mollview(mask_patch, hold=True, title="DR1 South mask")
plt.axes(ax3)
hp.mollview(mask_dr1, hold=True, title="DR1 mask")

plt.show()
fig.savefig(f'/home/jaimerzp/Desktop/mixing_mat_plots/masks.pdf', bbox_inches='tight')

: 

: 

## Cls

In [ ]:
## full-sky
path = "./dummy/"
full_sky_cls = {}
for i in range(1, n+1):
    print(f"Loading sim {i}", end='\r')
    full_sky_cls[i] = heracles.read(path+f"/cls/cls_data_{i}_lmax_{lmax}.fits")
full_sky_cqs = heracles.binned(full_sky_cls, ledges)

## half-sky
path = "./rr2/"
rr2_cls = {}
rr2_inv_cls = {}
rr2_nu_cls = {}
rr2_pols_cls = {}
for i in range(1, n+1):
    print(f"Loading sim {i}", end='\r')
    rr2_cls[i] = heracles.read(path+f"/cls/cls_data_{i}_lmax_{lmax}.fits")
    rr2_inv_cls[i] = heracles.read(path+f"/cls_inv/cls_data_inv_{i}_l1max_{lmax}_l2max_{lmax_mask}.fits")
    rr2_nu_cls[i] = heracles.read(path+f"/cls_nu/cls_data_nu_{i}_l1max_{lmax}_l2max_{lmax_mask}.fits")
    rr2_pols_cls[i] = heracles.read(path+f"/cls_pols/cls_data_pols_{i}_lmax_{lmax}.fits")

rr2_nmt_cqs = {}
for i in range(1, n+1):
    print(f"Loading sim {i}", end='\r')
    rr2_nmt_cqs[i] = heracles.read(path+f"/cls_nmt/cqs_data_nmt_np_{i}_lmax_{lmax}.fits")
rr2_cqs = heracles.binned(rr2_cls, ledges)
rr2_inv_cqs = heracles.binned(rr2_inv_cls, ledges)
rr2_nu_cqs = heracles.binned(rr2_nu_cls, ledges)
rr2_pols_cqs = heracles.binned(rr2_pols_cls, ledges)

## Planck
path = "./dr1/"
dr1_cls = {}
dr1_inv_cls = {}
dr1_nu_cls = {}
dr1_pols_cls = {}
for i in range(1, n+1):
    print(f"Loading sim {i}", end='\r')
    dr1_cls[i] = heracles.read(path+f"/cls/cls_data_{i}_lmax_{lmax}.fits")
    dr1_inv_cls[i] = heracles.read(path+f"/cls_inv/cls_data_inv_{i}_l1max_{lmax}_l2max_{lmax_mask}.fits")
    dr1_nu_cls[i] = heracles.read(path+f"/cls_nu/cls_data_nu_{i}_l1max_{lmax}_l2max_{lmax_mask}.fits")
    dr1_pols_cls[i] = heracles.read(path+f"/cls_pols/cls_data_pols_{i}_lmax_{lmax}.fits")

dr1_nmt_cqs = {}
for i in range(1, n+1):
    print(f"Loading sim {i}", end='\r')
    dr1_nmt_cqs[i] = heracles.read(path+f"/cls_nmt/cqs_data_nmt_np_{i}_lmax_{lmax}.fits")
dr1_cqs = heracles.binned(dr1_cls, ledges)
dr1_inv_cqs = heracles.binned(dr1_inv_cls, ledges)
dr1_nu_cqs = heracles.binned(dr1_nu_cls, ledges)
dr1_pols_cqs = heracles.binned(dr1_pols_cls, ledges)

## patch
path = "./patch/"
patch_cls = {}
patch_inv_cls = {}
patch_nu_cls = {}
patch_pols_cls = {}
for i in range(1, n+1):
    print(f"Loading sim {i}", end='\r')
    patch_cls[i] = heracles.read(path+f"/cls/cls_data_{i}_lmax_{lmax}.fits")
    patch_inv_cls[i] = heracles.read(path+f"/cls_inv/cls_data_inv_{i}_l1max_{lmax}_l2max_{lmax_mask}.fits")
    patch_nu_cls[i] = heracles.read(path+f"/cls_nu/cls_data_nu_{i}_l1max_{lmax}_l2max_{lmax_mask}.fits")
    patch_pols_cls[i] = heracles.read(path+f"/cls_pols/cls_data_pols_{i}_lmax_{lmax}.fits")

patch_nmt_cqs = {}
for i in range(1, n+1):
    print(f"Loading sim {i}", end='\r')
    patch_nmt_cqs[i] = heracles.read(path+f"/cls_nmt/cqs_data_nmt_np_{i}_lmax_{lmax}.fits")
patch_cqs = heracles.binned(patch_cls, ledges)
patch_inv_cqs = heracles.binned(patch_inv_cls, ledges)
patch_nu_cqs = heracles.binned(patch_nu_cls, ledges)
patch_pols_cqs = heracles.binned(patch_pols_cls, ledges)

: 

: 

In [ ]:
theory_cls = heracles.read(f"lognormal_sims/cls_theory.fits")
ls = np.arange(lmax+1)
fl = -np.sqrt((ls+2)*(ls+1)*ls*(ls-1))
fl /= np.clip(ls*(ls+1), 1, None)

_theory_cls = {}
_theory_cls[("POS", "POS", 1, 1)] = heracles.Result(theory_cls["W1xW1"].array[:lmax+1], ell=ls)

c = np.zeros((2, 2, lmax+1))
c[0, 0, :] = theory_cls["W2xW2"].array[:lmax+1] * fl**2
_theory_cls[("SHE", "SHE", 1, 1)] = heracles.Result(c)

c = np.zeros((2, lmax+1))
c[0, :] = theory_cls["W1xW2"].array[:lmax+1] * fl
_theory_cls[("POS", "SHE", 1, 1)] = heracles.Result(c)

_theory_cqs = heracles.binned(_theory_cls, ledges)


: 

: 

In [ ]:
def get_cls_mean(cls_dict):
    n_keys = list(cls_dict.keys())
    f_keys = list(cls_dict[n_keys[0]].keys())
    cls_mean = {}
    for f_key in f_keys:
        cl = np.mean([cls_dict[i][f_key] for i in n_keys], axis=0)
        cls_mean[f_key] = heracles.Result(cl, axis=cls_dict[n_keys[0]][f_key].axis)
    return cls_mean

def get_cls_std(cls_dict):
    n_keys = list(cls_dict.keys())
    f_keys = list(cls_dict[n_keys[0]].keys())
    cls_std = {}
    for f_key in f_keys:
        cl = np.std([cls_dict[i][f_key] for i in n_keys], axis=0)
        cls_std[f_key] = heracles.Result(cl, axis=cls_dict[n_keys[0]][f_key].axis)
    return cls_std

: 

: 

In [ ]:
## Full Sky means
full_sky_cls_m, full_sky_cls_s = get_cls_mean(full_sky_cls), get_cls_std(full_sky_cls)
full_sky_cqs_m, full_sky_cqs_s = get_cls_mean(full_sky_cqs), get_cls_std(full_sky_cqs)

## RR2 means
rr2_cls_m, rr2_cls_s = get_cls_mean(rr2_cls), get_cls_std(rr2_cls)
rr2_inv_cls_m, rr2_inv_cls_s = get_cls_mean(rr2_inv_cls), get_cls_std(rr2_inv_cls)
rr2_nu_cls_m, rr2_nu_cls_s = get_cls_mean(rr2_nu_cls), get_cls_std(rr2_nu_cls)
rr2_pols_cls_m, rr2_pols_cls_s = get_cls_mean(rr2_pols_cls), get_cls_std(rr2_pols_cls)

rr2_cqs_m, rr2_cqs_s = get_cls_mean(rr2_cqs), get_cls_std(rr2_cqs)
rr2_inv_cqs_m, rr2_inv_cqs_s = get_cls_mean(rr2_inv_cqs), get_cls_std(rr2_inv_cqs)
rr2_nu_cqs_m, rr2_nu_cqs_s = get_cls_mean(rr2_nu_cqs), get_cls_std(rr2_nu_cqs)
rr2_pols_cqs_m, rr2_pols_cqs_s = get_cls_mean(rr2_pols_cqs), get_cls_std(rr2_pols_cqs)
rr2_nmt_cqs_m, rr2_nmt_cqs_s = get_cls_mean(rr2_nmt_cqs), get_cls_std(rr2_nmt_cqs)

## DR1 South means
patch_cls_m, patch_cls_s = get_cls_mean(patch_cls), get_cls_std(patch_cls)
patch_inv_cls_m, patch_inv_cls_s = get_cls_mean(patch_inv_cls), get_cls_std(patch_inv_cls)
patch_nu_cls_m, patch_nu_cls_s = get_cls_mean(patch_nu_cls), get_cls_std(patch_nu_cls)
patch_pols_cls_m, patch_pols_cls_s = get_cls_mean(patch_pols_cls), get_cls_std(patch_pols_cls)

patch_cqs_m, patch_cqs_s = get_cls_mean(patch_cqs), get_cls_std(patch_cqs)
patch_inv_cqs_m, patch_inv_cqs_s = get_cls_mean(patch_inv_cqs), get_cls_std(patch_inv_cqs)
patch_nu_cqs_m, patch_nu_cqs_s = get_cls_mean(patch_nu_cqs), get_cls_std(patch_nu_cqs)
patch_pols_cqs_m, patch_pols_cqs_s = get_cls_mean(patch_pols_cqs), get_cls_std(patch_pols_cqs)
patch_nmt_cqs_m, patch_nmt_cqs_s = get_cls_mean(patch_nmt_cqs), get_cls_std(patch_nmt_cqs)

## DR1 means
dr1_cls_m, dr1_cls_s = get_cls_mean(dr1_cls), get_cls_std(dr1_cls)
dr1_inv_cls_m, dr1_inv_cls_s = get_cls_mean(dr1_inv_cls), get_cls_std(dr1_inv_cls)
dr1_nu_cls_m, dr1_nu_cls_s = get_cls_mean(dr1_nu_cls), get_cls_std(dr1_nu_cls)
dr1_pols_cls_m, dr1_pols_cls_s = get_cls_mean(dr1_pols_cls), get_cls_std(dr1_pols_cls)

dr1_cqs_m, dr1_cqs_s = get_cls_mean(dr1_cqs), get_cls_std(dr1_cqs)
dr1_inv_cqs_m, dr1_inv_cqs_s = get_cls_mean(dr1_inv_cqs), get_cls_std(dr1_inv_cqs)
dr1_nu_cqs_m, dr1_nu_cqs_s = get_cls_mean(dr1_nu_cqs), get_cls_std(dr1_nu_cqs)
dr1_pols_cqs_m, dr1_pols_cqs_s = get_cls_mean(dr1_pols_cqs), get_cls_std(dr1_pols_cqs)
dr1_nmt_cqs_m, dr1_nmt_cqs_s = get_cls_mean(dr1_nmt_cqs), get_cls_std(dr1_nmt_cqs)

: 

: 

In [ ]:
# Create a figure with 1 row and 3 columns of subplots
fig, axs = plt.subplots(3, 3, figsize=(15, 12))  # 3 rows, 3 columns

##########
# half-sky
##########

# POS POS
axs[0, 0].plot(l[2:], l[2:]*rr2_cls_m[("POS", "POS", 1, 1)].array[2:], color='C0', label='PP')
axs[0, 0].plot(l[2:], l[2:]*_theory_cls[("POS", "POS", 1, 1)].array[2:], color='C0', linestyle='--')
axs[0, 0].legend()
axs[0, 0].set_title("POS-POS")
axs[0, 0].set_xscale('log')
axs[0, 0].set_yscale('log')
axs[0, 0].set_xlabel("l")
axs[0, 0].set_ylabel(r"RR2 - $C_\ell$")

# POS SHE
axs[0, 1].plot(l[2:], l[2:]*rr2_cls_m[("POS", "SHE", 1, 1)].array[0][2:], color='C0', label='PE')
axs[0, 1].plot(l[2:], l[2:]*_theory_cls[("POS", "SHE", 1, 1)].array[0][2:], color='C0', linestyle='--')
axs[0, 1].plot(l[2:], l[2:]*rr2_cls_m[("POS", "SHE", 1, 1)].array[1][2:], color='C1', label='PB')
axs[0, 1].plot(l[2:], l[2:]*_theory_cls[("POS", "SHE", 1, 1)].array[1][2:], color='C1', linestyle='--')
axs[0, 1].legend()
axs[0, 1].set_title("POS-SHE")
axs[0, 1].set_xscale('log')
axs[0, 1].set_yscale('symlog')
axs[0, 1].set_xlabel("l")

# SHE SHE
axs[0, 2].plot(l[2:], l[2:]*rr2_cls_m[("SHE", "SHE", 1, 1)].array[0, 0][2:], color='C0', label='EE')
axs[0, 2].plot(l[2:], l[2:]*_theory_cls[("SHE", "SHE", 1, 1)].array[0, 0][2:], color='C0', linestyle='--')
axs[0, 2].plot(l[2:], l[2:]*rr2_cls_m[("SHE", "SHE", 1, 1)].array[0, 1][2:], color='C2', label='EB')
axs[0, 2].plot(l[2:], l[2:]*_theory_cls[("SHE", "SHE", 1, 1)].array[0, 1][2:], color='C2', linestyle='--')
axs[0, 2].plot(l[2:], l[2:]*rr2_cls_m[("SHE", "SHE", 1, 1)].array[1, 1][2:], color='C1', label='BB')
axs[0, 2].plot(l[2:], l[2:]*_theory_cls[("SHE", "SHE", 1, 1)].array[1, 1][2:], color='C1', linestyle='--')
axs[0, 2].legend()
axs[0, 2].set_title("SHE-SHE")
axs[0, 2].set_xscale('log')
axs[0, 2].set_yscale('log')
#axs[0, 2].set_yscale('symlog', linthresh=1e00)
axs[0, 2].set_xlabel("l")

##########
# patch
##########

# POS POS
axs[1, 0].plot(l[2:], l[2:]*patch_cls_m[("POS", "POS", 1, 1)].array[2:], color='C0', label='PP')
axs[1, 0].plot(l[2:], l[2:]*_theory_cls[("POS", "POS", 1, 1)].array[2:], color='C0', linestyle='--')
axs[1, 0].legend()
axs[1, 0].set_title("POS-POS")
axs[1, 0].set_xscale('log')
axs[1, 0].set_yscale('log')
axs[1, 0].set_xlabel("l")
axs[1, 0].set_ylabel(r"DR1 South - $C_\ell$")

# POS SHE
axs[1, 1].plot(l[2:], l[2:]*patch_cls_m[("POS", "SHE", 1, 1)].array[0][2:], color='C0', label='PE')
axs[1, 1].plot(l[2:], l[2:]*_theory_cls[("POS", "SHE", 1, 1)].array[0][2:], color='C0', linestyle='--')
axs[1, 1].plot(l[2:], l[2:]*patch_cls_m[("POS", "SHE", 1, 1)].array[1][2:], color='C1', label='PB')
axs[1, 1].plot(l[2:], l[2:]*_theory_cls[("POS", "SHE", 1, 1)].array[1][2:], color='C1', linestyle='--')
axs[1, 1].legend()
axs[1, 1].set_title("POS-SHE")
axs[1, 1].set_xscale('log')
axs[1, 1].set_yscale('symlog')
axs[1, 1].set_xlabel("l")

# SHE SHE
axs[1, 2].plot(l[2:], l[2:]*patch_cls_m[("SHE", "SHE", 1, 1)].array[0, 0][2:], color='C0', label='EE')
axs[1, 2].plot(l[2:], l[2:]*_theory_cls[("SHE", "SHE", 1, 1)].array[0, 0][2:], color='C0', linestyle='--')
axs[1, 2].plot(l[2:], l[2:]*patch_cls_m[("SHE", "SHE", 1, 1)].array[0, 1][2:], color='C2', label='EB')
axs[1, 2].plot(l[2:], l[2:]*_theory_cls[("SHE", "SHE", 1, 1)].array[0, 1][2:], color='C2', linestyle='--')
axs[1, 2].plot(l[2:], l[2:]*patch_cls_m[("SHE", "SHE", 1, 1)].array[1, 1][2:], color='C1', label='BB')
axs[1, 2].plot(l[2:], l[2:]*_theory_cls[("SHE", "SHE", 1, 1)].array[1, 1][2:], color='C1', linestyle='--')
axs[1, 2].legend()
axs[1, 2].set_title("SHE-SHE")
axs[1, 2].set_xscale('log')
axs[1, 2].set_yscale('log')
#axs[0, 2].set_yscale('symlog', linthresh=1e00)
axs[1, 2].set_xlabel("l")

##########
# planck
##########

# POS POS
axs[2, 0].plot(l[2:], l[2:]*dr1_cls_m[("POS", "POS", 1, 1)].array[2:], color='C0', label='PP')
axs[2, 0].plot(l[2:], l[2:]*_theory_cls[("POS", "POS", 1, 1)].array[2:], color='C0', linestyle='--')
axs[2, 0].legend()
axs[2, 0].set_title("POS-POS")
axs[2, 0].set_xscale('log')
axs[2, 0].set_yscale('log')
axs[2, 0].set_xlabel("l")
axs[2, 0].set_ylabel(r"DR1 - $C_\ell$")

# POS SHE
axs[2, 1].plot(l[2:], l[2:]*dr1_cls_m[("POS", "SHE", 1, 1)].array[0][2:], color='C0', label='PE')
axs[2, 1].plot(l[2:], l[2:]*_theory_cls[("POS", "SHE", 1, 1)].array[0][2:], color='C0', linestyle='--')
axs[2, 1].plot(l[2:], l[2:]*dr1_cls_m[("POS", "SHE", 1, 1)].array[1][2:], color='C1', label='PB')
axs[2, 1].plot(l[2:], l[2:]*_theory_cls[("POS", "SHE", 1, 1)].array[1][2:], color='C1', linestyle='--')
axs[2, 1].legend()
axs[2, 1].set_title("POS-SHE")
axs[2, 1].set_xscale('log')
axs[2, 1].set_yscale('symlog')
axs[2, 1].set_xlabel("l")

# SHE SHE
axs[2, 2].plot(l[2:], l[2:]*dr1_cls_m[("SHE", "SHE", 1, 1)].array[0, 0][2:], color='C0', label='EE')
axs[2, 2].plot(l[2:], l[2:]*_theory_cls[("SHE", "SHE", 1, 1)].array[0, 0][2:], color='C0', linestyle='--')
axs[2, 2].plot(l[2:], l[2:]*dr1_cls_m[("SHE", "SHE", 1, 1)].array[0, 1][2:], color='C2', label='EB')
axs[2, 2].plot(l[2:], l[2:]*_theory_cls[("SHE", "SHE", 1, 1)].array[0, 1][2:], color='C2', linestyle='--')
axs[2, 2].plot(l[2:], l[2:]*dr1_cls_m[("SHE", "SHE", 1, 1)].array[1, 1][2:], color='C1', label='BB')
axs[2, 2].plot(l[2:], l[2:]*_theory_cls[("SHE", "SHE", 1, 1)].array[1, 1][2:], color='C1', linestyle='--')
axs[2, 2].legend()
axs[2, 2].set_title("SHE-SHE")
axs[2, 2].set_xscale('log')
axs[2, 2].set_yscale('log')
#axs[0, 2].set_yscale('symlog', linthresh=1e00)
axs[2, 2].set_xlabel("l")

# Adjust layout
plt.tight_layout()
plt.show()

: 

: 

In [ ]:
# Create a figure with 1 row and 3 columns of subplots
fig, axs = plt.subplots(3, 3, figsize=(15, 12))  # 1 row, 3 columns

##########
## half-sky
##########
fsky=fsky_rr2
cls_m=rr2_cls_m

# POS POS
axs[0, 0].plot(l[2:], (cls_m[("POS", "POS", 1, 1)].array[2:]-fsky*_theory_cls[("POS", "POS", 1, 1)].array[2:])/(fsky*_theory_cls[("POS", "POS", 1, 1)].array[2:]), color='C0', label='PP')
axs[0, 0].legend()
axs[0, 0].set_title("POS-POS")
axs[0, 0].set_xscale('log')
axs[0, 0].set_xlabel("l")
axs[0, 0].set_ylabel("C_l")

# POS SHE
axs[0, 1].plot(l[2:], (cls_m[("POS", "SHE", 1, 1)].array[0][2:]-fsky_rr2*_theory_cls[("POS", "SHE", 1, 1)].array[0][2:])/(fsky_rr2*_theory_cls[("POS", "SHE", 1, 1)].array[0][2:]), color='C0', label='PE')
axs[0, 1].plot(l[2:], (cls_m[("POS", "SHE", 1, 1)].array[1][2:]-fsky_rr2*_theory_cls[("POS", "SHE", 1, 1)].array[1][2:])/(fsky_rr2*_theory_cls[("POS", "SHE", 1, 1)].array[1][2:]), color='C1', label='PB')
axs[0, 1].legend()
axs[0, 1].set_title("POS-SHE")
axs[0, 1].set_xscale('log')
axs[0, 1].set_xlabel("l")

# SHE SHE
axs[0, 2].plot(l[2:], (cls_m[("SHE", "SHE", 1, 1)].array[0, 0][2:]-fsky*_theory_cls[("SHE", "SHE", 1, 1)].array[0, 0][2:])/(fsky*_theory_cls[("SHE", "SHE", 1, 1)].array[0, 0][2:]), color='C0', label='EE')
axs[0, 2].plot(l[2:], (cls_m[("SHE", "SHE", 1, 1)].array[0, 1][2:]-fsky*_theory_cls[("SHE", "SHE", 1, 1)].array[0, 1][2:])/(fsky*_theory_cls[("SHE", "SHE", 1, 1)].array[0, 1][2:]), color='C2', label='EB')
axs[0, 2].plot(l[2:], (cls_m[("SHE", "SHE", 1, 1)].array[1, 1][2:]-fsky*_theory_cls[("SHE", "SHE", 1, 1)].array[1, 1][2:])/(fsky*_theory_cls[("SHE", "SHE", 1, 1)].array[1, 1][2:]), color='C1', label='BB')
axs[0, 2].legend()
axs[0, 2].set_title("SHE-SHE")
axs[0, 2].set_xscale('log')
#axs[0, 2].set_yscale('symlog', linthresh=1e00)
axs[0, 2].set_xlabel("l")

##########
## patch
##########
fsky=fsky_patch
cls_m=patch_cls_m

# POS POS
axs[1, 0].plot(l[2:], (cls_m[("POS", "POS", 1, 1)].array[2:]-fsky*_theory_cls[("POS", "POS", 1, 1)].array[2:])/(fsky*_theory_cls[("POS", "POS", 1, 1)].array[2:]), color='C0', label='PP')
axs[1, 0].legend()
axs[1, 0].set_title("POS-POS")
axs[1, 0].set_xscale('log')
axs[1, 0].set_xlabel("l")
axs[1, 0].set_ylabel("C_l")

# POS SHE
axs[1, 1].plot(l[2:], (cls_m[("POS", "SHE", 1, 1)].array[0][2:]-fsky_rr2*_theory_cls[("POS", "SHE", 1, 1)].array[0][2:])/(fsky_rr2*_theory_cls[("POS", "SHE", 1, 1)].array[0][2:]), color='C0', label='PE')
axs[1, 1].plot(l[2:], (cls_m[("POS", "SHE", 1, 1)].array[1][2:]-fsky_rr2*_theory_cls[("POS", "SHE", 1, 1)].array[1][2:])/(fsky_rr2*_theory_cls[("POS", "SHE", 1, 1)].array[1][2:]), color='C1', label='PB')
axs[1, 1].legend()
axs[1, 1].set_title("POS-SHE")
axs[1, 1].set_xscale('log')
axs[1, 1].set_xlabel("l")

# SHE SHE
axs[1, 2].plot(l[2:], (cls_m[("SHE", "SHE", 1, 1)].array[0, 0][2:]-fsky*_theory_cls[("SHE", "SHE", 1, 1)].array[0, 0][2:])/(fsky*_theory_cls[("SHE", "SHE", 1, 1)].array[0, 0][2:]), color='C0', label='EE')
axs[1, 2].plot(l[2:], (cls_m[("SHE", "SHE", 1, 1)].array[0, 1][2:]-fsky*_theory_cls[("SHE", "SHE", 1, 1)].array[0, 1][2:])/(fsky*_theory_cls[("SHE", "SHE", 1, 1)].array[0, 1][2:]), color='C2', label='EB')
axs[1, 2].plot(l[2:], (cls_m[("SHE", "SHE", 1, 1)].array[1, 1][2:]-fsky*_theory_cls[("SHE", "SHE", 1, 1)].array[1, 1][2:])/(fsky*_theory_cls[("SHE", "SHE", 1, 1)].array[1, 1][2:]), color='C1', label='BB')
axs[1, 2].legend()
axs[1, 2].set_title("SHE-SHE")
axs[1, 2].set_xscale('log')
#axs[0, 2].set_yscale('symlog', linthresh=1e00)
axs[1, 2].set_xlabel("l")

##########
## planck
##########
fsky=fsky_dr1
cls_m=dr1_cls_m

# POS POS
axs[2, 0].plot(l[2:], (cls_m[("POS", "POS", 1, 1)].array[2:]-fsky*_theory_cls[("POS", "POS", 1, 1)].array[2:])/(fsky*_theory_cls[("POS", "POS", 1, 1)].array[2:]), color='C0', label='PP')
axs[2, 0].legend()
axs[2, 0].set_title("POS-POS")
axs[2, 0].set_xscale('log')
axs[2, 0].set_xlabel("l")
axs[2, 0].set_ylabel("C_l")

# POS SHE
axs[2, 1].plot(l[2:], (cls_m[("POS", "SHE", 1, 1)].array[0][2:]-fsky_rr2*_theory_cls[("POS", "SHE", 1, 1)].array[0][2:])/(fsky_rr2*_theory_cls[("POS", "SHE", 1, 1)].array[0][2:]), color='C0', label='PE')
axs[2, 1].plot(l[2:], (cls_m[("POS", "SHE", 1, 1)].array[1][2:]-fsky_rr2*_theory_cls[("POS", "SHE", 1, 1)].array[1][2:])/(fsky_rr2*_theory_cls[("POS", "SHE", 1, 1)].array[1][2:]), color='C1', label='PB')
axs[2, 1].legend()
axs[2, 1].set_title("POS-SHE")
axs[2, 1].set_xscale('log')
axs[2, 1].set_xlabel("l")

# SHE SHE
axs[2, 2].plot(l[2:], (cls_m[("SHE", "SHE", 1, 1)].array[0, 0][2:]-fsky*_theory_cls[("SHE", "SHE", 1, 1)].array[0, 0][2:])/(fsky*_theory_cls[("SHE", "SHE", 1, 1)].array[0, 0][2:]), color='C0', label='EE')
axs[2, 2].plot(l[2:], (cls_m[("SHE", "SHE", 1, 1)].array[0, 1][2:]-fsky*_theory_cls[("SHE", "SHE", 1, 1)].array[0, 1][2:])/(fsky*_theory_cls[("SHE", "SHE", 1, 1)].array[0, 1][2:]), color='C2', label='EB')
axs[2, 2].plot(l[2:], (cls_m[("SHE", "SHE", 1, 1)].array[1, 1][2:]-fsky*_theory_cls[("SHE", "SHE", 1, 1)].array[1, 1][2:])/(fsky*_theory_cls[("SHE", "SHE", 1, 1)].array[1, 1][2:]), color='C1', label='BB')
axs[2, 2].legend()
axs[2, 2].set_title("SHE-SHE")
axs[2, 2].set_xscale('log')
#axs[0, 2].set_yscale('symlog', linthresh=1e00)
axs[2, 2].set_xlabel("l")

# Adjust layout
plt.tight_layout()
plt.show()

: 

: 

In [ ]:
xvals, _ = heracles.transforms._cached_gauss_legendre(lmax_mask+1)

patch_mask_cls = heracles.read(f"./patch/cls/cls_mask_lmax_{lmax_mask}.fits")
rr2d_mask_cls = heracles.read(f"./rr2/cls/cls_mask_lmax_{lmax_mask}.fits")
dr1_mask_cls = heracles.read(f"./dr1/cls/cls_mask_lmax_{lmax_mask}.fits")

rr2_mask_corr = {}
patch_mask_corr = {}
dr1_mask_corr = {}
rr2_mask_corr = {}
dr1_mask_corr = {}
rr2_mask_corr_log = {}
patch_mask_corr_log = {}
dr1_mask_corr_log = {}
rr2_mask_corr_log = {}
dr1_mask_corr_log = {}
m_keys = list(dr1_mask_cls.keys())
for m_key in m_keys:
    _patch_m = patch_mask_cls[m_key]
    _rr2d_m = rr2d_mask_cls[m_key]
    _dr1_m = dr1_mask_cls[m_key]
    # Compute wm
    _patch_wm = heracles.transforms.cl2corr(_patch_m)
    _rr2d_wm = heracles.transforms.cl2corr(_rr2d_m)
    _dr1_wm = heracles.transforms.cl2corr(_dr1_m)
    # Save
    patch_mask_corr[m_key] = 1/_patch_wm.T[0]
    rr2_mask_corr[m_key] = 1/_rr2d_wm.T[0]
    dr1_mask_corr[m_key] = 1/_dr1_wm.T[0]
    # Smooth wm
    rcond = 1e-3
    _patch_wm = _patch_wm.T[0]
    cutoff = rcond #* np.max(np.abs(_patch_wm))
    #_patch_wm = np.array([1/wi if abs(wi) > cutoff else 0 for wi in _patch_wm])
    _patch_wm *= heracles.unmixing.logistic(np.log10(abs(_patch_wm)), x0=-4, k=50)
    _patch_wm = 1/_patch_wm
    _rr2d_wm = _rr2d_wm.T[0]
    cutoff = rcond #* np.max(np.abs(_rr2d_wm))
    #_rr2d_wm = np.array([1/wi if abs(wi) > cutoff else 0 for wi in _rr2d_wm])
    _rr2d_wm *= heracles.unmixing.logistic(np.log10(abs(_rr2d_wm)), x0=-4, k=50)
    _rr2d_wm = 1/_rr2d_wm
    _dr1_wm = _dr1_wm.T[0]
    cutoff = rcond #* np.max(np.abs(_dr1_wm))
    #_dr1_wm = np.array([1/wi if abs(wi) > cutoff else 0 for wi in _dr1_wm])
    _dr1_wm *= heracles.unmixing.logistic(np.log10(abs(_dr1_wm)), x0=-4, k=50)
    _dr1_wm = 1/_dr1_wm
    # Save
    patch_mask_corr_log[m_key] = _patch_wm
    rr2_mask_corr_log[m_key] = _rr2d_wm
    dr1_mask_corr_log[m_key] = _dr1_wm



: 

: 

In [ ]:
#fig, ax = plt.subplots(figsize=(8,6))
f = patch_mask_corr["VIS", "VIS", 1, 1]-patch_mask_corr["VIS", "VIS", 1, 1][-1] 
plt.plot(xvals[::-1], f[::-1], 'C2-', alpha=0.4, label='DR1 South')
f = rr2_mask_corr["VIS", "VIS", 1, 1]-rr2_mask_corr["VIS", "VIS", 1, 1][-1] 
plt.plot(xvals[::-1], f[::-1], 'C3-', alpha=0.4, label='RR2')
f = dr1_mask_corr["VIS", "VIS", 1, 1]-dr1_mask_corr["VIS", "VIS", 1, 1][-1] 
plt.plot(xvals[::-1], f[::-1], 'C4-', alpha=0.4, label='DR1')

f = patch_mask_corr_log["VIS", "VIS", 1, 1]-patch_mask_corr_log["VIS", "VIS", 1, 1][-1]
plt.plot(xvals, f, 'C2--')
f = rr2_mask_corr_log["VIS", "VIS", 1, 1]-rr2_mask_corr_log["VIS", "VIS", 1, 1][-1]
plt.plot(xvals, f, 'C3--')
f = dr1_mask_corr_log["VIS", "VIS", 1, 1]-dr1_mask_corr_log["VIS", "VIS", 1, 1][-1]
plt.plot(xvals, f, 'C4--')

plt.legend()
plt.yscale('symlog')
plt.ylim(0, 1e12)
plt.xlim(-1., 1)
plt.xlabel(r'$cos(\theta)$')
plt.ylabel(r'$\frac{1}{wm(\theta)}$', rotation=0)

: 

: 

## MixMats

In [ ]:
rr2_mixmat = heracles.read(f"/home/jaimerzp/Documents/UCL/GLASS_cov_challenge/rr2/mixmat_l1max_{lmax}_l2max_{lmax_mask}.fits")
rr2_unmixmat = heracles.read(f"/home/jaimerzp/Documents/UCL/GLASS_cov_challenge/rr2/unmixmat_l1max_{lmax}_l2max_{lmax_mask}.fits")
rr2_inv_mixmat = heracles.read(f"/home/jaimerzp/Documents/UCL/GLASS_cov_challenge/rr2/inv_mixmat_l1max_{lmax}_l2max_{lmax_mask}.fits")

dr1_mixmat = heracles.read(f"/home/jaimerzp/Documents/UCL/GLASS_cov_challenge/dr1/mixmat_l1max_{lmax}_l2max_{lmax_mask}.fits")
dr1_unmixmat = heracles.read(f"/home/jaimerzp/Documents/UCL/GLASS_cov_challenge/dr1/unmixmat_l1max_{lmax}_l2max_{lmax_mask}.fits")
dr1_inv_mixmat = heracles.read(f"/home/jaimerzp/Documents/UCL/GLASS_cov_challenge/dr1/inv_mixmat_l1max_{lmax}_l2max_{lmax_mask}.fits")

patch_mixmat = heracles.read(f"/home/jaimerzp/Documents/UCL/GLASS_cov_challenge/patch/mixmat_l1max_{lmax}_l2max_{lmax_mask}.fits")
patch_unmixmat = heracles.read(f"/home/jaimerzp/Documents/UCL/GLASS_cov_challenge/patch/unmixmat_l1max_{lmax}_l2max_{lmax_mask}.fits")
patch_inv_mixmat = heracles.read(f"/home/jaimerzp/Documents/UCL/GLASS_cov_challenge/patch/inv_mixmat_l1max_{lmax}_l2max_{lmax_mask}.fits")


: 

: 

In [ ]:
def svd_pinv(A, x0=-3.5, rcond=1e-2):
    """`
    Compute the Moore–Penrose pseudoinverse of a matrix A
    using its Singular Value Decomposition (SVD).

    Parameters
    ----------
    A : (m, n) array_like
        Input matrix to be pseudoinverted.
    rcond : float, optional
        Cutoff for small singular values.
        Singular values smaller than rcond * max(s) are set to zero.

    Returns
    -------
    A_pinv : (n, m) ndarray
        The pseudoinverse of A.
    """
    # Step 1: Compute SVD
    U, s, Vt = np.linalg.svd(A, full_matrices=False)

    # Step 2: Invert singular values with tolerance
    cutoff = rcond * np.max(s)
    _s_inv = np.array([1/si if si > cutoff else 0 for si in s])

    #_s = s * heracles.unmixing.logistic(np.log10(s), x0=x0) 
    s_inv = 1/s

    # Step 3: Reconstruct pseudoinverse
    # A^+ = V * Σ^+ * U^T
    #A_pinv = (Vt.T * s_inv) @ U.T
    return s_inv, _s_inv

: 

: 

In [ ]:
xvals, _ = heracles.transforms._cached_gauss_legendre(lmax_mask+1)

patch_mask_cls = heracles.read(f"./patch/cls/cls_mask_lmax_{lmax_mask}.fits")
rr2d_mask_cls = heracles.read(f"./rr2/cls/cls_mask_lmax_{lmax_mask}.fits")
dr1_mask_cls = heracles.read(f"./dr1/cls/cls_mask_lmax_{lmax_mask}.fits")

rr2_mask_corr = {}
patch_mask_corr = {}
dr1_mask_corr = {}
rr2_mask_corr = {}
dr1_mask_corr = {}
rr2_mask_corr_log = {}
patch_mask_corr_log = {}
dr1_mask_corr_log = {}
rr2_mask_corr_log = {}
dr1_mask_corr_log = {}
m_keys = list(dr1_mask_cls.keys())
for m_key in m_keys:
    _patch_m = patch_mask_cls[m_key]
    _rr2_m = rr2d_mask_cls[m_key]
    _dr1_m = dr1_mask_cls[m_key]
    # Compute wm
    _patch_wm = heracles.transforms.cl2corr(_patch_m).T[0]
    _rr2_wm = heracles.transforms.cl2corr(_rr2_m).T[0]
    _dr1_wm = heracles.transforms.cl2corr(_dr1_m).T[0]
    # Save
    patch_mask_corr[m_key] = 1/_patch_wm
    rr2_mask_corr[m_key] = 1/_rr2_wm
    dr1_mask_corr[m_key] = 1/_dr1_wm
    # Smooth wm
    rcond = 1e-2
    cutoff = rcond * np.max(np.abs(_patch_wm))
    #_patch_wm = np.array([1/wi if abs(wi) > cutoff else 0 for wi in _patch_wm])
    _patch_wm *= heracles.unmixing.logistic(np.log10(abs(_patch_wm)), x0=np.log10(cutoff))
    _patch_wm = 1/_patch_wm  
    cutoff = rcond * np.max(np.abs(_rr2_wm))
    #_rr2_wm = np.array([1/wi if abs(wi) > cutoff else 0 for wi in _rr2_wm])
    _rr2_wm *= heracles.unmixing.logistic(np.log10(abs(_rr2_wm)), x0=np.log10(cutoff))
    _rr2_wm = 1/_rr2_wm
    cutoff = rcond * np.max(np.abs(_dr1_wm))
    #_dr1_wm = np.array([1/wi if abs(wi) > cutoff else 0 for wi in _dr1_wm])
    _dr1_wm *= heracles.unmixing.logistic(np.log10(abs(_dr1_wm)), x0=np.log10(cutoff))
    _dr1_wm = 1/_dr1_wm 
    # Save
    patch_mask_corr_log[m_key] = _patch_wm
    rr2_mask_corr_log[m_key] = _rr2_wm
    dr1_mask_corr_log[m_key] = _dr1_wm

: 

: 

In [ ]:
rr2_s_inv, rr2__s_inv = svd_pinv(rr2_mixmat["POS", "POS", 1, 1].array, x0=-3, rcond=1e-2)
dr1_s_inv, dr1__s_inv = svd_pinv(dr1_mixmat["POS", "POS", 1, 1].array, x0=-3, rcond=1e-2)
patch_s_inv, patch__s_inv = svd_pinv(patch_mixmat["POS", "POS", 1, 1].array, x0=-3, rcond=1e-2)

: 

: 

In [ ]:
f = rr2_s_inv[::-1]
plt.plot(f/f[-1], 'C2-', alpha=0.6, label='RR2')
f = dr1_s_inv[::-1]
plt.plot(f/f[-1], 'C3-', alpha=0.6, label='DR1')
f = patch_s_inv[::-1]
plt.plot(f/f[-1], 'C4-', alpha=0.6, label='Patch')

f = rr2__s_inv[::-1]
plt.plot(f/f[-1], 'C2--', alpha=0.6)
f = dr1__s_inv[::-1]
plt.plot(f/f[-1], 'C3--', alpha=0.6)
f = patch__s_inv[::-1]
plt.plot(f/f[-1], 'C4--', alpha=0.6)

plt.xlabel(r'Singular value index')
plt.ylabel(r'$1/s$', rotation=0)
plt.yscale('log')
#plt.ylim(1e-6, 1e9)
plt.legend()

: 

: 

In [ ]:
fig, axs = plt.subplots(3, 4, figsize=(16, 8))
plt.subplots_adjust(hspace = 0.0, wspace = 0.1)

#######
## half-sky
#######

# Open the FITS file
binv_mixmat = heracles.binned(rr2_inv_mixmat, ledges)
bmixmat = heracles.binned(rr2_mixmat, ledges)
bmixmatb = {}
for key in bmixmat.keys():
    bmixmatb[key] = heracles.binned(heracles.Result(bmixmat[key].array, axis=bmixmat[key].axis[0]+1, ell=l_mask), ledges)
bmixmatb = heracles.binned(bmixmatb, ledges)
inv_bmixmatb = heracles.invert_mixing_matrix(bmixmatb, rtol=0.01)
# Kernels
inv_kernel = rr2_inv_mixmat["POS", "POS", 1, 1].array @ rr2_mixmat["POS", "POS", 1, 1].array
inv_kernel = heracles.binned(inv_kernel, ledges).array.T
nmt_kernel = inv_bmixmatb["POS", "POS", 1, 1].array @ bmixmat["POS", "POS", 1, 1].array
pols_kernel = fits.open(f"/home/jaimerzp/Documents/UCL/GLASS_cov_challenge/rr2/cls_pols/kernels_pols_1_l1max_{lmax}.fits")[0].data[0, :, :].T
pols_kernel = heracles.Result(pols_kernel[:lmax+1, :lmax+1], axis=(0,), ell=l)
pols_kernel = heracles.binned(pols_kernel, ledges).array
nu_kernel = heracles.Result(rr2_unmixmat["POS", "POS", 1, 1].array.T @ rr2_mixmat["POS", "POS", 1, 1].array, axis=(0,), ell=l_mask)
nu_kernel = heracles.binned(nu_kernel, ledges).array

im0= axs[0, 0].imshow(np.log10(np.abs(inv_kernel)), cmap='seismic', aspect='auto', vmin=-10, vmax=1)
axs[0, 0].get_xaxis().set_ticks([])
axs[0, 0].set_ylabel("RR2")
axs[0, 0].set_title('Inversion', y=1.05)

axs[0, 1].imshow(np.log10(np.abs(nmt_kernel)), cmap='seismic', aspect='auto', vmin=-10, vmax=1)
axs[0, 1].get_yaxis().set_ticklabels([])
axs[0, 1].get_xaxis().set_ticks([])
axs[0, 1].set_title('NaMaster', y=1.05)

axs[0, 2].imshow(np.log10(np.abs(nu_kernel)), cmap='seismic', aspect='auto', vmin=-10, vmax=1)
axs[0, 2].get_yaxis().set_ticklabels([])
axs[0, 2].get_xaxis().set_ticks([])
axs[0, 2].set_title('Natural Unmixing', y=1.05)

axs[0, 3].imshow(np.log10(np.abs(pols_kernel)), cmap='seismic', aspect='auto', vmin=-10, vmax=1)
axs[0, 3].get_yaxis().set_ticklabels([])
axs[0, 3].get_xaxis().set_ticks([])
axs[0, 3].set_title('PolSpice', y=1.05)


#######
## patch
#######

# Open the FITS file
binv_mixmat = heracles.binned(patch_inv_mixmat, ledges)
# invert binned mixing matrix
bmixmat = heracles.binned(patch_mixmat, ledges)
bmixmatb = {}
for key in bmixmat.keys():
    bmixmatb[key] = heracles.binned(heracles.Result(bmixmat[key].array, axis=bmixmat[key].axis[0]+1, ell=l_mask), ledges)
bmixmatb = heracles.binned(bmixmatb, ledges)
inv_bmixmatb = heracles.invert_mixing_matrix(bmixmatb, rtol=0.01)
# Kernels
inv_kernel = patch_inv_mixmat["POS", "POS", 1, 1].array @ patch_mixmat["POS", "POS", 1, 1].array
inv_kernel = heracles.binned(inv_kernel, ledges).array.T
nmt_kernel = inv_bmixmatb["POS", "POS", 1, 1].array @ bmixmat["POS", "POS", 1, 1].array
pols_kernel = fits.open(f"/home/jaimerzp/Documents/UCL/GLASS_cov_challenge/patch/cls_pols/kernels_pols_1_l1max_{lmax}.fits")[0].data[0, :, :].T
pols_kernel = heracles.Result(pols_kernel[:lmax+1, :lmax+1], axis=(0,), ell=l)
pols_kernel = heracles.binned(pols_kernel, ledges).array
nu_kernel = heracles.Result(patch_unmixmat["POS", "POS", 1, 1].array.T @ patch_mixmat["POS", "POS", 1, 1].array, axis=(0,), ell=l_mask)
nu_kernel = heracles.binned(nu_kernel, ledges).array

axs[1, 0].imshow(np.log10(np.abs(inv_kernel)), cmap='seismic', aspect='auto', vmin=-10, vmax=1)
axs[1, 0].get_xaxis().set_ticklabels([])
axs[1, 0].set_ylabel("DR1 South")

axs[1, 1].imshow(np.log10(np.abs(nmt_kernel)), cmap='seismic', aspect='auto', vmin=-10, vmax=1)
axs[1, 1].get_xaxis().set_ticklabels([])
axs[1, 1].get_yaxis().set_ticklabels([])

axs[1, 2].imshow(np.log10(np.abs(nu_kernel)), cmap='seismic', aspect='auto', vmin=-10, vmax=1)
axs[1, 2].get_yaxis().set_ticklabels([])

axs[1, 3].imshow(np.log10(np.abs(pols_kernel)), cmap='seismic', aspect='auto', vmin=-10, vmax=1)
axs[1, 3].get_xaxis().set_ticklabels([])
axs[1, 3].get_yaxis().set_ticklabels([])

#######
## planck
#######

# Open the FITS file
# invert binned mixing matrix
bmixmat = heracles.binned(dr1_mixmat, ledges)
bmixmatb = {}
for key in bmixmat.keys():
    bmixmatb[key] = heracles.binned(heracles.Result(bmixmat[key].array, axis=bmixmat[key].axis[0]+1, ell=l_mask), ledges)
bmixmatb = heracles.binned(bmixmatb, ledges)
inv_bmixmatb = heracles.invert_mixing_matrix(bmixmatb, rtol=0.01)
# Kernels
inv_kernel = patch_inv_mixmat["POS", "POS", 1, 1].array @ dr1_mixmat["POS", "POS", 1, 1].array
inv_kernel = heracles.binned(inv_kernel, ledges).array.T
nmt_kernel = inv_bmixmatb["POS", "POS", 1, 1].array @ bmixmat["POS", "POS", 1, 1].array
pols_kernel = fits.open(f"/home/jaimerzp/Documents/UCL/GLASS_cov_challenge/dr1/cls_pols/kernels_pols_1_l1max_{lmax}.fits")[0].data[0, :, :].T
pols_kernel = heracles.Result(pols_kernel[:lmax+1, :lmax+1], axis=(0,), ell=l)
pols_kernel = heracles.binned(pols_kernel, ledges).array
nu_kernel = heracles.Result(dr1_unmixmat["POS", "POS", 1, 1].array.T @ dr1_mixmat["POS", "POS", 1, 1].array, axis=(0,), ell=l_mask)
nu_kernel = heracles.binned(nu_kernel, ledges).array

axs[2, 0].imshow(np.log10(np.abs(inv_kernel)), cmap='seismic', aspect='auto', vmin=-10, vmax=1)
axs[2, 0].set_ylabel("DR1")
axs[2, 1].imshow(np.log10(np.abs(nmt_kernel)), cmap='seismic', aspect='auto', vmin=-10, vmax=1)
axs[2, 1].get_yaxis().set_ticklabels([])
axs[2, 2].imshow(np.log10(np.abs(nu_kernel)), cmap='seismic', aspect='auto', vmin=-10, vmax=1)
axs[2, 3].imshow(np.log10(np.abs(pols_kernel)), cmap='seismic', aspect='auto', vmin=-10, vmax=1)

fig.subplots_adjust(right=0.8)
cbar_ax = fig.add_axes([0.82, 0.15, 0.02, 0.7])
fig.colorbar(im0, cax=cbar_ax)
plt.show()
fig.savefig(f'/home/jaimerzp/Desktop/mixing_mat_plots/kernel_compo_{mask_type}.pdf', bbox_inches='tight')

: 

: 

In [ ]:
fsky_rr2=np.sum(mask_rr2)/len(mask_rr2)
fsky_dr1=np.sum(mask_dr1)/len(mask_dr1)
fsky_patch=np.sum(mask_patch)/len(mask_patch)

: 

: 

In [ ]:
## Full Sky means
full_sky_cls_m = get_cls_mean(full_sky_cls)
full_sky_cqs_m = get_cls_mean(full_sky_cqs)

## Half Sky means
rr2_inv_cls_m = get_cls_mean(rr2_inv_cls)
rr2_nu_cls_m = get_cls_mean(rr2_nu_cls)
rr2_pols_cls_m = get_cls_mean(rr2_pols_cls)

rr2_inv_cqs_m = get_cls_mean(rr2_inv_cqs)
rr2_nu_cqs_m = get_cls_mean(rr2_nu_cqs)
rr2_pols_cqs_m = get_cls_mean(rr2_pols_cqs)
rr2_nmt_cqs_m = get_cls_mean(rr2_nmt_cqs)

## Planck means
dr1_inv_cls_m = get_cls_mean(dr1_inv_cls)
dr1_nu_cls_m = get_cls_mean(dr1_nu_cls)
dr1_pols_cls_m = get_cls_mean(dr1_pols_cls)

dr1_inv_cqs_m = get_cls_mean(dr1_inv_cqs)
dr1_nu_cqs_m = get_cls_mean(dr1_nu_cqs)
dr1_pols_cqs_m = get_cls_mean(dr1_pols_cqs)
dr1_nmt_cqs_m = get_cls_mean(dr1_nmt_cqs)

## patch means
patch_inv_cls_m = get_cls_mean(patch_inv_cls)
patch_nu_cls_m = get_cls_mean(patch_nu_cls)
patch_pols_cls_m = get_cls_mean(patch_pols_cls)

patch_inv_cqs_m = get_cls_mean(patch_inv_cqs)
patch_nu_cqs_m = get_cls_mean(patch_nu_cqs)
patch_pols_cqs_m = get_cls_mean(patch_pols_cqs)
patch_nmt_cqs_m = get_cls_mean(patch_nmt_cqs)

: 

: 

In [ ]:
## full_sky covariances
full_sky_ensemble_cov = heracles.read(f"dummy/covs/cov_cls_l1max_{lmax}_l2max_{lmax_mask}.fits")
full_sky_ensemble_covqq = heracles.read(f"dummy/covs/cov_cqs_l1max_{lmax}_l2max_{lmax_mask}.fits")
full_sky_flat_ensemble_cov = dices.flatten(full_sky_ensemble_covqq)

## half_sky covariances
rr2_ensemble_cov = heracles.read(f"rr2/covs/cov_cls_l1max_{lmax}_l2max_{lmax_mask}.fits")
rr2_ensemble_covqq = heracles.read(f"rr2/covs/cov_cqs_l1max_{lmax}_l2max_{lmax_mask}.fits")

rr2_inv_ensemble_cov = heracles.read(f"rr2/covs/cov_inv_cls_l1max_{lmax}_l2max_{lmax_mask}.fits")
rr2_nu_ensemble_cov = heracles.read(f"rr2/covs/cov_nu_cls_l1max_{lmax}_l2max_{lmax_mask}.fits")
rr2_pols_ensemble_cov = heracles.read(f"rr2/covs/cov_pols_cls_l1max_{lmax}_l2max_{lmax_mask}.fits")

rr2_inv_ensemble_covqq = heracles.read(f"rr2/covs/cov_inv_cqs_l1max_{lmax}_l2max_{lmax_mask}.fits")
rr2_nu_ensemble_covqq = heracles.read(f"rr2/covs/cov_nu_cqs_l1max_{lmax}_l2max_{lmax_mask}.fits")
rr2_nmt_ensemble_covqq = heracles.read(f"rr2/covs/cov_nmt_cqs_l1max_{lmax}_l2max_{lmax_mask}.fits")
rr2_pols_ensemble_covqq = heracles.read(f"rr2/covs/cov_pols_cqs_l1max_{lmax}_l2max_{lmax_mask}.fits")

rr2_flat_ensemble_cov = dices.flatten(rr2_ensemble_covqq)
rr2_flat_inv_ensemble_cov = dices.flatten(rr2_inv_ensemble_covqq)
rr2_flat_nu_ensemble_cov = dices.flatten(rr2_nu_ensemble_covqq)
rr2_flat_pols_ensemble_cov = dices.flatten(rr2_pols_ensemble_covqq)
rr2_flat_nmt_ensemble_cov = dices.flatten(rr2_nmt_ensemble_covqq)

rr2_flat_ensemble_corr = rr2_flat_ensemble_cov / np.sqrt(
    np.diag(rr2_flat_ensemble_cov)[:, None] * np.diag(rr2_flat_ensemble_cov)[None, :]
)
rr2_flat_inv_ensemble_corr = rr2_flat_inv_ensemble_cov / np.sqrt(
    np.diag(rr2_flat_inv_ensemble_cov)[:, None] * np.diag(rr2_flat_inv_ensemble_cov)[None, :]
)
rr2_flat_nu_ensemble_corr = rr2_flat_nu_ensemble_cov / np.sqrt(
    np.diag(rr2_flat_nu_ensemble_cov)[:, None] * np.diag(rr2_flat_nu_ensemble_cov)[None, :]
)
rr2_flat_pols_ensemble_corr = rr2_flat_pols_ensemble_cov / np.sqrt(
    np.diag(rr2_flat_pols_ensemble_cov)[:, None] * np.diag(rr2_flat_pols_ensemble_cov)[None, :]
)
rr2_flat_nmt_ensemble_corr = rr2_flat_nmt_ensemble_cov / np.sqrt(
    np.diag(rr2_flat_nmt_ensemble_cov)[:, None] * np.diag(rr2_flat_nmt_ensemble_cov)[None, :]
)


## Planck covariances
dr1_ensemble_cov = heracles.read(f"dr1/covs/cov_cls_l1max_{lmax}_l2max_{lmax_mask}.fits")
dr1_ensemble_covqq = heracles.read(f"dr1/covs/cov_cqs_l1max_{lmax}_l2max_{lmax_mask}.fits")

dr1_inv_ensemble_cov = heracles.read(f"dr1/covs/cov_inv_cls_l1max_{lmax}_l2max_{lmax_mask}.fits")
dr1_nu_ensemble_cov = heracles.read(f"dr1/covs/cov_nu_cls_l1max_{lmax}_l2max_{lmax_mask}.fits")
dr1_pols_ensemble_cov = heracles.read(f"dr1/covs/cov_pols_cls_l1max_{lmax}_l2max_{lmax_mask}.fits")

dr1_inv_ensemble_covqq = heracles.read(f"dr1/covs/cov_inv_cqs_l1max_{lmax}_l2max_{lmax_mask}.fits")
dr1_nu_ensemble_covqq = heracles.read(f"dr1/covs/cov_nu_cqs_l1max_{lmax}_l2max_{lmax_mask}.fits")
dr1_nmt_ensemble_covqq = heracles.read(f"dr1/covs/cov_nmt_cqs_l1max_{lmax}_l2max_{lmax_mask}.fits")
dr1_pols_ensemble_covqq = heracles.read(f"dr1/covs/cov_pols_cqs_l1max_{lmax}_l2max_{lmax_mask}.fits")

dr1_flat_ensemble_cov = dices.flatten(dr1_ensemble_covqq)
dr1_flat_inv_ensemble_cov = dices.flatten(dr1_inv_ensemble_covqq)
dr1_flat_nu_ensemble_cov = dices.flatten(dr1_nu_ensemble_covqq)
dr1_flat_pols_ensemble_cov = dices.flatten(dr1_pols_ensemble_covqq)
dr1_flat_nmt_ensemble_cov = dices.flatten(dr1_nmt_ensemble_covqq)

dr1_flat_ensemble_corr = dr1_flat_ensemble_cov / np.sqrt(
    np.diag(dr1_flat_ensemble_cov)[:, None] * np.diag(dr1_flat_ensemble_cov)[None, :]
)
dr1_flat_inv_ensemble_corr = dr1_flat_inv_ensemble_cov / np.sqrt(
    np.diag(dr1_flat_inv_ensemble_cov)[:, None] * np.diag(dr1_flat_inv_ensemble_cov)[None, :]
)
dr1_flat_nu_ensemble_corr = dr1_flat_nu_ensemble_cov / np.sqrt(
    np.diag(dr1_flat_nu_ensemble_cov)[:, None] * np.diag(dr1_flat_nu_ensemble_cov)[None, :]
)
dr1_flat_pols_ensemble_corr = dr1_flat_pols_ensemble_cov / np.sqrt(
    np.diag(dr1_flat_pols_ensemble_cov)[:, None] * np.diag(dr1_flat_pols_ensemble_cov)[None, :]
)
dr1_flat_nmt_ensemble_corr = dr1_flat_nmt_ensemble_cov / np.sqrt(
    np.diag(dr1_flat_nmt_ensemble_cov)[:, None] * np.diag(dr1_flat_nmt_ensemble_cov)[None, :]
)

## patch covariances
patch_ensemble_cov = heracles.read(f"patch/covs/cov_cls_l1max_{lmax}_l2max_{lmax_mask}.fits")
patch_ensemble_covqq = heracles.read(f"patch/covs/cov_cqs_l1max_{lmax}_l2max_{lmax_mask}.fits")

patch_inv_ensemble_cov = heracles.read(f"patch/covs/cov_inv_cls_l1max_{lmax}_l2max_{lmax_mask}.fits")
patch_nu_ensemble_cov = heracles.read(f"patch/covs/cov_nu_cls_l1max_{lmax}_l2max_{lmax_mask}.fits")
patch_pols_ensemble_cov = heracles.read(f"patch/covs/cov_pols_cls_l1max_{lmax}_l2max_{lmax_mask}.fits")

patch_inv_ensemble_covqq = heracles.read(f"patch/covs/cov_inv_cqs_l1max_{lmax}_l2max_{lmax_mask}.fits")
patch_nu_ensemble_covqq = heracles.read(f"patch/covs/cov_nu_cqs_l1max_{lmax}_l2max_{lmax_mask}.fits")
patch_nmt_ensemble_covqq = heracles.read(f"patch/covs/cov_nmt_cqs_l1max_{lmax}_l2max_{lmax_mask}.fits")
patch_pols_ensemble_covqq = heracles.read(f"patch/covs/cov_pols_cqs_l1max_{lmax}_l2max_{lmax_mask}.fits")

patch_flat_ensemble_cov = dices.flatten(patch_ensemble_covqq)
patch_flat_inv_ensemble_cov = dices.flatten(patch_inv_ensemble_covqq)
patch_flat_nu_ensemble_cov = dices.flatten(patch_nu_ensemble_covqq)
patch_flat_pols_ensemble_cov = dices.flatten(patch_pols_ensemble_covqq)
patch_flat_nmt_ensemble_cov = dices.flatten(patch_nmt_ensemble_covqq)

patch_flat_ensemble_corr = patch_flat_ensemble_cov / np.sqrt(
    np.diag(patch_flat_ensemble_cov)[:, None] * np.diag(patch_flat_ensemble_cov)[None, :]
)
patch_flat_inv_ensemble_corr = patch_flat_inv_ensemble_cov / np.sqrt(
    np.diag(patch_flat_inv_ensemble_cov)[:, None] * np.diag(patch_flat_inv_ensemble_cov)[None, :]
)
patch_flat_nu_ensemble_corr = patch_flat_nu_ensemble_cov / np.sqrt(
    np.diag(patch_flat_nu_ensemble_cov)[:, None] * np.diag(patch_flat_nu_ensemble_cov)[None, :]
)
patch_flat_pols_ensemble_corr = patch_flat_pols_ensemble_cov / np.sqrt(
    np.diag(patch_flat_pols_ensemble_cov)[:, None] * np.diag(patch_flat_pols_ensemble_cov)[None, :]
)
patch_flat_nmt_ensemble_corr = patch_flat_nmt_ensemble_cov / np.sqrt(
    np.diag(patch_flat_nmt_ensemble_cov)[:, None] * np.diag(patch_flat_nmt_ensemble_cov)[None, :]
)

: 

: 

In [ ]:
def combine_matrices(A, B):
    """
    Combines two square matrices:
    - Lower triangle (including diagonal) from A
    - Upper triangle from B
    """
    # Make sure A and B are numpy arrays
    A = np.array(A)
    B = np.array(B)

    # Create an empty matrix
    C = np.zeros_like(A)

    # Use masks for lower and upper triangles
    lower_mask = np.tri(A.shape[0], dtype=bool)  # i >= j
    upper_mask = ~lower_mask                     # i < j

    # Assign values
    C[lower_mask] = A[lower_mask]
    C[upper_mask] = B[upper_mask]

    return C

: 

: 

In [ ]:
fig, axes = plt.subplots(3, 4, figsize=(12, 9))
plt.subplots_adjust(hspace = 0.0, wspace = 0.0)


# Flattened Inverse Covariance
comb_corr = combine_matrices(rr2_flat_ensemble_corr, dr1_flat_inv_ensemble_corr)
axes[0, 0].imshow(comb_corr, cmap="seismic", vmin=-1, vmax=1)
axes[0, 0].set_ylabel("RR2")
axes[0, 0].set_title("Inverse")
axes[0, 0].get_xaxis().set_ticks([])
# Flattened NaMaster covariance
comb_corr = combine_matrices(rr2_flat_ensemble_corr, dr1_flat_nmt_ensemble_corr)
axes[0, 1].imshow(comb_corr, cmap="seismic", vmin=-1, vmax=1)
axes[0, 1].set_title("NaMaster")
axes[0, 1].get_yaxis().set_ticks([])
axes[0, 1].get_xaxis().set_ticks([])
# Flattened Nautural Unmixing Covariance
comb_corr = combine_matrices(rr2_flat_ensemble_corr, dr1_flat_nu_ensemble_corr)
axes[0, 2].imshow(comb_corr, cmap="seismic", vmin=-1, vmax=1)
axes[0, 2].set_title("Natural Unmixing")
axes[0, 2].get_yaxis().set_ticks([])
axes[0, 2].get_xaxis().set_ticks([])
# Flattened PolSpice Covariance
comb_corr = combine_matrices(rr2_flat_ensemble_corr, dr1_flat_pols_ensemble_corr)
axes[0, 3].imshow(comb_corr, cmap="seismic", vmin=-1, vmax=1)
axes[0, 3].set_title("PolSpice")
axes[0, 3].get_yaxis().set_ticks([])
axes[0, 3].get_xaxis().set_ticks([])

# Flattened Inverse Covariance
comb_corr = combine_matrices(patch_flat_ensemble_corr, dr1_flat_inv_ensemble_corr)
axes[1, 0].imshow(comb_corr, cmap="seismic", vmin=-1, vmax=1)
axes[1, 0].set_ylabel("DR1 South")
axes[1, 0].get_xaxis().set_ticks([])
# Flattened NaMaster covariance
comb_corr = combine_matrices(patch_flat_ensemble_corr, dr1_flat_nmt_ensemble_corr)
axes[1, 1].imshow(comb_corr, cmap="seismic", vmin=-1, vmax=1)
axes[1, 1].get_yaxis().set_ticks([])
axes[1, 1].get_xaxis().set_ticks([])
# Flattened Nautural Unmixing Covariance
comb_corr = combine_matrices(patch_flat_ensemble_corr, dr1_flat_nu_ensemble_corr)
axes[1, 2].imshow(comb_corr, cmap="seismic", vmin=-1, vmax=1)
axes[1, 2].get_yaxis().set_ticks([])
axes[1, 2].get_xaxis().set_ticks([])
# Flattened PolSpice Covariance
comb_corr = combine_matrices(patch_flat_ensemble_corr, dr1_flat_pols_ensemble_corr)
axes[1, 3].imshow(comb_corr, cmap="seismic", vmin=-1, vmax=1)
axes[1, 3].get_yaxis().set_ticks([])
axes[1, 3].get_xaxis().set_ticks([])

# Flattened Inverse Covariance
comb_corr = combine_matrices(dr1_flat_ensemble_corr, patch_flat_inv_ensemble_corr)
axes[2, 0].imshow(comb_corr, cmap="seismic", vmin=-1, vmax=1)
axes[2, 0].set_ylabel("DR1")
# Flattened NaMaster covariance
comb_corr = combine_matrices(dr1_flat_ensemble_corr, patch_flat_nmt_ensemble_corr)
axes[2, 1].imshow(comb_corr, cmap="seismic", vmin=-1, vmax=1)
axes[2, 1].get_yaxis().set_ticks([])
# Flattened Nautural Unmixing Covariance
comb_corr = combine_matrices(dr1_flat_ensemble_corr, patch_flat_nu_ensemble_corr)
axes[2, 2].imshow(comb_corr, cmap="seismic", vmin=-1, vmax=1)
axes[2, 2].get_yaxis().set_ticks([])
# Flattened PolSpice Covariance
comb_corr = combine_matrices(dr1_flat_ensemble_corr, patch_flat_pols_ensemble_corr)
axes[2, 3].imshow(comb_corr, cmap="seismic", vmin=-1, vmax=1)
axes[2, 3].get_yaxis().set_ticks([])

#plt.tight_layout()
plt.show()
fig.savefig(f'/home/jaimerzp/Desktop/mixing_mat_plots/corr_comp.pdf', bbox_inches='tight')

: 

: 

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(12, 4))
fig.subplots_adjust(wspace = 0.0,hspace = 0.0)

########
## half-sky
########
ens_err = np.sqrt(np.diag(rr2_flat_ensemble_cov))/fsky_rr2
inv_ens_err = np.sqrt(np.diag(rr2_flat_inv_ensemble_cov))
nmt_ens_err = np.sqrt(np.diag(rr2_flat_nmt_ensemble_cov))
nu_ens_err = np.sqrt(np.diag(rr2_flat_nu_ensemble_cov))
pols_ens_err = np.sqrt(np.diag(rr2_flat_pols_ensemble_cov))

axes[0].fill_between(np.arange(len(inv_ens_err)), inv_ens_err/ens_err, alpha=0.2)
axes[0].fill_between(np.arange(len(nmt_ens_err)), nmt_ens_err/ens_err, alpha=0.2)
axes[0].fill_between(np.arange(len(nu_ens_err)), nu_ens_err/ens_err, alpha=0.2)
axes[0].fill_between(np.arange(len(pols_ens_err)), pols_ens_err/ens_err, alpha=0.2)
axes[0].plot(inv_ens_err/ens_err, label='Inverse')
axes[0].plot(nmt_ens_err/ens_err, label='NaMaster')
axes[0].plot(nu_ens_err/ens_err, label='Natural Unmixing')
axes[0].plot(pols_ens_err/ens_err, label='PolSpice')
axes[0].axhline(0, color='k', linestyle='--', linewidth=0.5)
axes[0].set_ylabel(r'$\sigma_{\rm model}/\sigma_{\rm data}$')
axes[0].set_xlim(0, 120)
axes[0].set_ylim(1e-2, 5e3)
axes[0].set_yscale('log')
axes[0].set_title('RR2 Mask', y=0.9)
axes[0].set_xticks(np.arange(20, 121, 20))
axes[0].set_xticklabels(["PxP", "PxE", "PxB", "ExE", "ExB", "BxB"])


########
## Planck
########
ens_err = np.sqrt(np.diag(patch_flat_ensemble_cov))/fsky_patch
inv_ens_err = np.sqrt(np.diag(patch_flat_inv_ensemble_cov))
nmt_ens_err = np.sqrt(np.diag(patch_flat_nmt_ensemble_cov))
nu_ens_err = np.sqrt(np.diag(patch_flat_nu_ensemble_cov))
pols_ens_err = np.sqrt(np.diag(patch_flat_pols_ensemble_cov))

axes[1].fill_between(np.arange(len(inv_ens_err)), inv_ens_err/ens_err, alpha=0.2)
axes[1].fill_between(np.arange(len(nmt_ens_err)), nmt_ens_err/ens_err, alpha=0.2)
axes[1].fill_between(np.arange(len(nu_ens_err)), nu_ens_err/ens_err, alpha=0.2)
axes[1].fill_between(np.arange(len(pols_ens_err)), pols_ens_err/ens_err, alpha=0.2)
axes[1].plot(inv_ens_err/ens_err, label='Inverse')
axes[1].plot(nmt_ens_err/ens_err, label='NaMaster')
axes[1].plot(nu_ens_err/ens_err, label='Natural Unmixing')
axes[1].plot(pols_ens_err/ens_err, label='PolSpice')
axes[1].axhline(0, color='k', linestyle='--', linewidth=0.5)
axes[1].set_xlim(0, 120)
axes[1].set_yscale('log')
axes[1].set_yticklabels([])
axes[1].set_ylim(1e-2, 5e3)
axes[1].set_xticks(np.arange(20, 121, 20))
axes[1].set_xticklabels(["PxP", "PxE", "PxB", "ExE", "ExB", "BxB"])
axes[1].set_title('DR1 South Mask', y=0.9)

#######
## patch
#######
ens_err = np.sqrt(np.diag(dr1_flat_ensemble_cov))/fsky_dr1
inv_ens_err = np.sqrt(np.diag(dr1_flat_inv_ensemble_cov))
nmt_ens_err = np.sqrt(np.diag(dr1_flat_nmt_ensemble_cov))
nu_ens_err = np.sqrt(np.diag(dr1_flat_nu_ensemble_cov))
pols_ens_err = np.sqrt(np.diag(dr1_flat_pols_ensemble_cov))

axes[2].fill_between(np.arange(len(inv_ens_err)), inv_ens_err/ens_err, alpha=0.2)
axes[2].fill_between(np.arange(len(nmt_ens_err)), nmt_ens_err/ens_err, alpha=0.2)
axes[2].fill_between(np.arange(len(nu_ens_err)), nu_ens_err/ens_err, alpha=0.2)
axes[2].fill_between(np.arange(len(pols_ens_err)), pols_ens_err/ens_err, alpha=0.2)
axes[2].plot(inv_ens_err/ens_err, label='Inverse')
axes[2].plot(nmt_ens_err/ens_err, label='NaMaster')
axes[2].plot(nu_ens_err/ens_err, label='Natural Unmixing')
axes[2].plot(pols_ens_err/ens_err, label='PolSpice')
axes[2].axhline(0, color='k', linestyle='--', linewidth=0.5)
axes[2].set_xlim(0, 120)
axes[2].set_yscale('log')
axes[2].set_xticks(np.arange(20, 121, 20))
axes[2].set_yticklabels([])
axes[2].set_xticklabels(["PxP", "PxE", "PxB", "ExE", "ExB", "BxB"])
axes[2].set_title('DR1 Mask', y=0.9)
axes[2].set_ylim(1e-2, 5e3)
axes[2].legend()

plt.tight_layout()
plt.show()
fig.savefig(f'/home/jaimerzp/Desktop/mixing_mat_plots/cov_comp.pdf', bbox_inches='tight')

: 

: 

In [ ]:
fig, ax = plt.subplots(6, 4, figsize=(14, 8), 
                       gridspec_kw={"height_ratios": [3, 1, 3, 1, 3, 1]})
fig.subplots_adjust(left=0.0, bottom=0.0, right=1.0, top=1.0, wspace=0.15, hspace=0.0)

#########
## half-sky
#########
ensemble_covqq = rr2_ensemble_covqq
inv_ensemble_covqq = rr2_inv_ensemble_covqq
nmt_ensemble_covqq = rr2_nmt_ensemble_covqq
nu_ensemble_covqq = rr2_nu_ensemble_covqq
pols_ensemble_covqq = rr2_pols_ensemble_covqq

inv_cqs_m = rr2_inv_cqs_m
nmt_cqs_m = rr2_nmt_cqs_m
nu_cqs_m = rr2_nu_cqs_m
pols_cqs_m = rr2_pols_cqs_m

# POS
key = ("POS", "POS", 1, 1)
cov_key = ("POS", "POS", "POS", "POS", 1, 1, 1, 1)

cov = ensemble_covqq[cov_key]
i_cov = inv_ensemble_covqq[cov_key]
nmt_cov = nmt_ensemble_covqq[cov_key]
nu_cov = nu_ensemble_covqq[cov_key]
pp_cov = pols_ensemble_covqq[cov_key]

err = np.sqrt(np.diag(cov))/fsky_rr2
i_err = np.sqrt(np.diag(i_cov))
nmt_err = np.sqrt(np.diag(nmt_cov))
nu_err = np.sqrt(np.diag(nu_cov))
pp_err = np.sqrt(np.diag(pp_cov))

i_c = inv_cqs_m[key]
nmt_c = nmt_cqs_m[key]
nu_c = nu_cqs_m[key]
pp_c = pols_cqs_m[key]
t = _theory_cls[key]
t_itp = np.interp(lgrid, ls, t)


ax[0, 0].errorbar(
            lgrid, lgrid*i_c, yerr=lgrid*i_err, fmt=".", c="C0", lw=1.5, zorder=3.0, alpha=0.5,
            label="Inversion"
        )
ax[0, 0].errorbar(
    lgrid, lgrid*nmt_c, yerr=lgrid*nmt_err, fmt=".", c="C1", lw=1.5, zorder=3.0, alpha=0.5,
    label="NaMaster"
        )
ax[0, 0].errorbar(
    lgrid, lgrid*nu_c, yerr=lgrid*nu_err, fmt=".", c="C2", lw=1.5, zorder=3.0, alpha=0.5,
    label="Natural Unmixing"
        )
ax[0, 0].errorbar(
    lgrid, lgrid*pp_c, yerr=lgrid*pp_err, fmt=".", c="C3", lw=1.5, zorder=3.0, alpha=0.5,
    label="PolSpice"
    )
ax[0, 0].plot(ls[10:], ls[10:]*t[10:], c="k", lw=1.0, zorder=4.0)
ax[1, 0].axhline(0.0, c="k", lw=0.8, zorder=-1)
ax[1, 0].errorbar(
    lgrid,
    (i_c - t_itp) / err,
    yerr=np.abs(i_err / i_err),
    fmt=".",
    c="C0",
    lw=1.5,
    zorder=1.0,
    alpha=0.5,
)
ax[1, 0].errorbar(
    lgrid,
    (nmt_c - t_itp) / err,
    yerr=np.abs(nmt_err / err),
    fmt=".",
    c="C1",
    lw=1.5,
    zorder=1.0,
    alpha=0.5,
)
ax[1, 0].errorbar(
    lgrid,
    (nu_c - t_itp) / err,
    yerr=np.abs(nu_err / err),
    fmt=".",
    c="C2",
    lw=1.5,
    zorder=1.0,
    alpha=0.5,
)
ax[1, 0].errorbar(
    lgrid,
    (pp_c - t_itp) / err,
    yerr=np.abs(pp_err / err),
    fmt=".",
    c="C3", 
    lw=1.5,
    zorder=1.0,
    alpha=0.5,
)
ax[1, 0].axhline(0.0, c="k", lw=0.8, zorder=-1)

ax[0, 0].tick_params(axis="both", which="both", direction="in")

ax[0, 0].set_title("POSxPOS", y=0.85)
ax[0, 0].set_xscale("log")
ax[0, 0].set_xlim(7, lmax * 1.5)
ax[0, 0].set_ylabel("RR2 - $\ell C_\ell$")
ax[1, 0].set_yscale("symlog", linthresh=1e0)
ax[1, 0].set_ylim(-50, 50)
ax[1, 0].set_xscale("log")
ax[1, 0].set_xlim(7, lmax * 1.5)
ax[1, 0].set_xlabel("$\ell$")
ax[1, 0].set_ylabel(r"$\frac{\ell \Delta C_\ell}{\sigma_{C_\ell}}$")

# POS-E
key = ("POS", "SHE", 1, 1)
cov_key = ("POS", "SHE", "POS", "SHE", 1, 1, 1, 1)

cov = ensemble_covqq[cov_key][0, 0, :, :]
i_cov = inv_ensemble_covqq[cov_key][0, 0, :, :]
nmt_cov = nmt_ensemble_covqq[cov_key][0, 0, :, :]
nu_cov = nu_ensemble_covqq[cov_key][0, 0, :, :]
pp_cov = pols_ensemble_covqq[cov_key][0, 0, :, :]

err = np.sqrt(np.diag(cov))/fsky_rr2
i_err = np.sqrt(np.diag(i_cov))
nmt_err = np.sqrt(np.diag(nmt_cov))
nu_err = np.sqrt(np.diag(nu_cov))
pp_err = np.sqrt(np.diag(pp_cov))

i_c = inv_cqs_m[key][0, :]
nmt_c = nmt_cqs_m[key][0, :]
nu_c = nu_cqs_m[key][0, :]
pp_c = pols_cqs_m[key][0, :]
t = _theory_cls[key][0, :]
t_itp = np.interp(lgrid, ls, t)

ax[0, 1].errorbar(
            lgrid, lgrid*i_c, yerr=lgrid*i_err, fmt=".", c="C0", lw=1.5, zorder=3.0, alpha=0.5,
            label="Inversion"
        )
ax[0, 1].errorbar(
    lgrid, lgrid*nmt_c, yerr=lgrid*nmt_err, fmt=".", c="C1", lw=1.5, zorder=3.0, alpha=0.5,
    label="NaMaster"
        )
ax[0, 1].errorbar(
    lgrid, lgrid*nu_c, yerr=lgrid*nu_err, fmt=".", c="C2", lw=1.5, zorder=3.0, alpha=0.5,
    label="Natural Unmixing"
        )
ax[0, 1].errorbar(
    lgrid, lgrid*pp_c, yerr=lgrid*pp_err, fmt=".", c="C3", lw=1.5, zorder=3.0, alpha=0.5,
    label="PolSpice"
        )
ax[0, 1].plot(ls[10:], ls[10:]*t[10:], c="k", lw=1.0, zorder=4.0)
ax[1, 1].axhline(0.0, c="k", lw=0.8, zorder=-1)
ax[1, 1].errorbar(
    lgrid,
    (i_c - t_itp) / err,
    yerr=np.abs(i_err / err),
    fmt=".",
    c="C0",
    lw=1.5,
    zorder=1.0,
    alpha=0.5,
)
ax[1, 1].errorbar(
    lgrid,
    (nu_c - t_itp) / err,
    yerr=np.abs(nu_err / err),
    fmt=".",
    c="C1",
    lw=1.5,
    zorder=1.0,
    alpha=0.5,
)
ax[1, 1].errorbar(
    lgrid,
    (pp_c - t_itp) / err,
    yerr=np.abs(pp_err / err),
    fmt=".",
    c="C2",
    lw=1.5,
    zorder=1.0,
    alpha=0.5,
)
ax[0, 1].tick_params(axis="both", which="both", direction="in")
ax[1, 1].axhline(0.0, c="k", lw=0.8, zorder=-1)

ax[0, 1].set_title("POSxE", y=0.85)
ax[0, 1].set_xscale("log")
ax[0, 1].set_xlim(7, lmax * 1.5)
ax[1, 1].set_yscale("symlog", linthresh=1e0)
ax[1, 1].set_ylim(-50, 50)
ax[1, 1].set_xscale("log")
ax[1, 1].set_xlim(7, lmax * 1.5)
ax[1, 1].set_xlabel(r"$\ell$")

# EE
key = ("SHE", "SHE", 1, 1)
cov_key = ("SHE", "SHE", "SHE", "SHE", 1, 1, 1, 1)

cov = ensemble_covqq[cov_key][0, 0, 0, 0, :, :]
i_cov = inv_ensemble_covqq[cov_key][0, 0, 0, 0, :, :]
nmt_cov = nmt_ensemble_covqq[cov_key][0, 0, 0, 0, :, :]
nu_cov = nu_ensemble_covqq[cov_key][0, 0, 0, 0, :, :]
pp_cov = pols_ensemble_covqq[cov_key][0, 0, 0, 0, :, :]

err = np.sqrt(np.diag(cov))/fsky_rr2
i_err = np.sqrt(np.diag(i_cov))
nmt_err = np.sqrt(np.diag(nmt_cov))
nu_err = np.sqrt(np.diag(nu_cov))
pp_err = np.sqrt(np.diag(pp_cov))

i_c = inv_cqs_m[key][0, 0, :]
nmt_c = nmt_cqs_m[key][0, 0, :]
nu_c = nu_cqs_m[key][0, 0, :]
pp_c = pols_cqs_m[key][0, 0, :]
t = _theory_cls[key][0, 0, :]
t_itp = np.interp(lgrid, ls, t)

ax[0, 2].errorbar(
            lgrid, lgrid*i_c, yerr=lgrid*i_err, fmt=".", c="C0", lw=1.5, zorder=3.0, alpha=0.5,
            label="Inversion"
        )
ax[0, 2].errorbar(
    lgrid, lgrid*nmt_c, yerr=lgrid*nmt_err, fmt=".", c="C1", lw=1.5, zorder=3.0, alpha=0.5,
    label="NaMaster"
        )
ax[0, 2].errorbar(
    lgrid, lgrid*nu_c, yerr=lgrid*nu_err, fmt=".", c="C2", lw=1.5, zorder=3.0, alpha=0.5,
    label="Natural Unmixing"
        )
ax[0, 2].errorbar(
    lgrid, lgrid*pp_c, yerr=lgrid*pp_err, fmt=".", c="C3", lw=1.5, zorder=3.0, alpha=0.5,
    label="PolSpice"
        )
ax[0, 2].plot(ls[10:], ls[10:]*t[10:], c="k", lw=1.0, zorder=4.0)
ax[1, 2].axhline(0.0, c="k", lw=0.8, zorder=-1)
ax[1, 2].errorbar(
    lgrid,
    (i_c - t_itp) / err,
    yerr=np.abs(i_err / err),
    fmt=".",
    c="C0",
    lw=1.5,
    zorder=1.0,
    alpha=0.5,
)
ax[1, 2].errorbar(
    lgrid,
    (nu_c - t_itp) / err,
    yerr=np.abs(nu_err / err),
    fmt=".",
    c="C1",
    lw=1.5,
    zorder=1.0,
    alpha=0.5,
)
ax[1, 2].errorbar(
    lgrid,
    (pp_c - t_itp) / err,
    yerr=np.abs(pp_err / err),
    fmt=".",
    c="C2",
    lw=1.5,
    zorder=1.0,
    alpha=0.5,
)
ax[0, 2].set_title("ExE", y=0.85)
ax[1, 2].axhline(0.0, c="k", lw=0.8, zorder=-1)
ax[0, 2].tick_params(axis="both", which="both", direction="in")
ax[0, 2].set_xscale("log")
ax[0, 2].set_xlim(7, lmax * 1.5)
ax[1, 2].set_yscale("symlog", linthresh=1e0)
ax[1, 2].set_ylim(-50, 50)
ax[1, 2].set_xscale("log")
ax[1, 2].set_xlim(7, lmax * 1.5)
ax[1, 2].set_xlabel(r"$\ell$")

# BB
key = ("SHE", "SHE", 1, 1)
cov_key = ("SHE", "SHE", "SHE", "SHE", 1, 1, 1, 1)

cov = ensemble_covqq[cov_key][1, 1, 1, 1, :, :]
i_cov = inv_ensemble_covqq[cov_key][1, 1, 1, 1, :, :]
nmt_cov = nmt_ensemble_covqq[cov_key][1, 1, 1, 1, :, :]
nu_cov = nu_ensemble_covqq[cov_key][1, 1, 1, 1, :, :]
pp_cov = pols_ensemble_covqq[cov_key][1, 1, 1, 1, :, :]

err = np.sqrt(np.diag(cov))/fsky_rr2
i_err = np.sqrt(np.diag(i_cov))
nmt_err = np.sqrt(np.diag(nmt_cov))
nu_err = np.sqrt(np.diag(nu_cov))
pp_err = np.sqrt(np.diag(pp_cov))

i_c = inv_cqs_m[key][1, 1, :]
nmt_c = nmt_cqs_m[key][1, 1, :]
nu_c = nu_cqs_m[key][1, 1, :]
pp_c = pols_cqs_m[key][1, 1, :]
t = _theory_cls[key][1, 1, :]
t_itp = np.interp(lgrid, ls, t)


ax[0, 3].errorbar(
            lgrid, lgrid*i_c, yerr=lgrid*i_err, fmt=".", c="C0", lw=1.5, zorder=3.0, alpha=0.5,
            label="Inversion"
        )
ax[0, 3].errorbar(
    lgrid, lgrid*nmt_c, yerr=lgrid*nmt_err, fmt=".", c="C1", lw=1.5, zorder=3.0, alpha=0.5,
    label="NaMaster"
        )
ax[0, 3].errorbar(
    lgrid, lgrid*nu_c, yerr=lgrid*nu_err, fmt=".", c="C2", lw=1.5, zorder=3.0, alpha=0.5,
    label="Natural Unmixing"
        )
ax[0, 3].errorbar(
    lgrid, lgrid*pp_c, yerr=lgrid*pp_err, fmt=".", c="C3", lw=1.5, zorder=3.0, alpha=0.5,
    label="PolSpice"
    )
ax[0, 3].plot(ls[10:], ls[10:]*t[10:], c="k", lw=1.0, zorder=4.0)
ax[1, 3].axhline(0.0, c="k", lw=0.8, zorder=-1)
ax[1, 3].errorbar(
    lgrid,
    (i_c - t_itp) / err,
    yerr=np.abs(i_err / err),
    fmt=".",
    c="C0",
    lw=1.5,
    zorder=1.0,
    alpha=0.5,
)
ax[1, 3].errorbar(
    lgrid,
    (nmt_c - t_itp) / err,
    yerr=np.abs(nmt_err / err),
    fmt=".",
    c="C1",
    lw=1.5,
    zorder=1.0,
    alpha=0.5,
)
ax[1, 3].errorbar(
    lgrid,
    (nu_c - t_itp) / err,
    yerr=np.abs(nu_err / err),
    fmt=".",
    c="C2",
    lw=1.5,
    zorder=1.0,
    alpha=0.5,
)
ax[1, 3].errorbar(
    lgrid,
    (pp_c - t_itp) / err,
    yerr=np.abs(pp_err / err),
    fmt=".",
    c="C3", 
    lw=1.5,
    zorder=1.0,
    alpha=0.5,
)

ax[0, 3].legend()
ax[0, 3].tick_params(axis="both", which="both", direction="in")
ax[1, 3].axhline(0.0, c="k", lw=0.8, zorder=-1)
ax[0, 3].set_title("BxB", y=0.85)
ax[0, 3].set_xscale("log")
ax[0, 3].set_xlim(7, lmax * 1.5)
ax[1, 3].set_yscale("symlog", linthresh=1e0)
ax[1, 3].set_ylim(-50, 50)
ax[1, 3].set_xscale("log")
ax[1, 3].set_xlim(7, lmax * 1.5)
ax[1, 3].set_xlabel(r"$\ell$")


#########
## Planck
#########
ensemble_covqq = patch_ensemble_covqq
inv_ensemble_covqq = patch_inv_ensemble_covqq
nmt_ensemble_covqq = patch_nmt_ensemble_covqq
nu_ensemble_covqq = patch_nu_ensemble_covqq
pols_ensemble_covqq = patch_pols_ensemble_covqq

inv_cqs_m = patch_inv_cqs_m
nmt_cqs_m = patch_nmt_cqs_m
nu_cqs_m = patch_nu_cqs_m
pols_cqs_m = patch_pols_cqs_m

# POS
key = ("POS", "POS", 1, 1)
cov_key = ("POS", "POS", "POS", "POS", 1, 1, 1, 1)

cov = ensemble_covqq[cov_key]
i_cov = inv_ensemble_covqq[cov_key]
nmt_cov = nmt_ensemble_covqq[cov_key]
nu_cov = nu_ensemble_covqq[cov_key]
pp_cov = pols_ensemble_covqq[cov_key]

err = np.sqrt(np.diag(cov))/fsky_dr1
i_err = np.sqrt(np.diag(i_cov))
nmt_err = np.sqrt(np.diag(nmt_cov))
nu_err = np.sqrt(np.diag(nu_cov))
pp_err = np.sqrt(np.diag(pp_cov))

i_c = inv_cqs_m[key]
nmt_c = nmt_cqs_m[key]
nu_c = nu_cqs_m[key]
pp_c = pols_cqs_m[key]
t = _theory_cls[key]
t_itp = np.interp(lgrid, ls, t)


ax[2, 0].errorbar(
            lgrid, lgrid*i_c, yerr=lgrid*i_err, fmt=".", c="C0", lw=1.5, zorder=3.0, alpha=0.5,
            label="Inversion"
        )
ax[2, 0].errorbar(
    lgrid, lgrid*nmt_c, yerr=lgrid*nmt_err, fmt=".", c="C1", lw=1.5, zorder=3.0, alpha=0.5,
    label="NaMaster"
        )
ax[2, 0].errorbar(
    lgrid, lgrid*nu_c, yerr=lgrid*nu_err, fmt=".", c="C2", lw=1.5, zorder=3.0, alpha=0.5,
    label="Natural Unmixing"
        )
ax[2, 0].errorbar(
    lgrid, lgrid*pp_c, yerr=lgrid*pp_err, fmt=".", c="C3", lw=1.5, zorder=3.0, alpha=0.5,
    label="PolSpice"
    )
ax[2, 0].plot(ls[10:], ls[10:]*t[10:], c="k", lw=1.0, zorder=4.0)
ax[3, 0].axhline(0.0, c="k", lw=0.8, zorder=-1)
ax[3, 0].errorbar(
    lgrid,
    (i_c - t_itp) / err,
    yerr=np.abs(i_err / err),
    fmt=".",
    c="C0",
    lw=1.5,
    zorder=1.0,
    alpha=0.5,
)
ax[3, 0].errorbar(
    lgrid,
    (nmt_c - t_itp) / err,
    yerr=np.abs(nmt_err / err),
    fmt=".",
    c="C1",
    lw=1.5,
    zorder=1.0,
    alpha=0.5,
)
ax[3, 0].errorbar(
    lgrid,
    (nu_c - t_itp) / err,
    yerr=np.abs(nu_err / err),
    fmt=".",
    c="C2",
    lw=1.5,
    zorder=1.0,
    alpha=0.5,
)
ax[3, 0].errorbar(
    lgrid,
    (pp_c - t_itp) / err,
    yerr=np.abs(pp_err / err),
    fmt=".",
    c="C3", 
    lw=1.5,
    zorder=1.0,
    alpha=0.5,
)
ax[3, 0].axhline(0.0, c="k", lw=0.8, zorder=-1)

ax[2, 0].tick_params(axis="both", which="both", direction="in")

ax[2, 0].set_xscale("log")
ax[2, 0].set_xlim(7, lmax * 1.5)
ax[2, 0].set_ylabel("DR1 South - $\ell C_\ell$")
ax[3, 0].set_yscale("symlog", linthresh=1e0)
ax[3, 0].set_ylim(-50, 50)
ax[3, 0].set_xscale("log")
ax[3, 0].set_xlim(7, lmax * 1.5)
ax[3, 0].set_xlabel("$\ell$")
ax[3, 0].set_ylabel(r"$\frac{\ell \Delta C_\ell}{\sigma_{C_\ell}}$")

# POS-E
key = ("POS", "SHE", 1, 1)
cov_key = ("POS", "SHE", "POS", "SHE", 1, 1, 1, 1)

cov = ensemble_covqq[cov_key][0, 0, :, :]
i_cov = inv_ensemble_covqq[cov_key][0, 0, :, :]
nmt_cov = nmt_ensemble_covqq[cov_key][0, 0, :, :]
nu_cov = nu_ensemble_covqq[cov_key][0, 0, :, :]
pp_cov = pols_ensemble_covqq[cov_key][0, 0, :, :]

err = np.sqrt(np.diag(cov))/fsky_dr1
i_err = np.sqrt(np.diag(i_cov))
nmt_err = np.sqrt(np.diag(nmt_cov))
nu_err = np.sqrt(np.diag(nu_cov))
pp_err = np.sqrt(np.diag(pp_cov))

i_c = inv_cqs_m[key][0, :]
nmt_c = nmt_cqs_m[key][0, :]
nu_c = nu_cqs_m[key][0, :]
pp_c = pols_cqs_m[key][0, :]
t = _theory_cls[key][0, :]
t_itp = np.interp(lgrid, ls, t)

ax[2, 1].errorbar(
            lgrid, lgrid*i_c, yerr=lgrid*i_err, fmt=".", c="C0", lw=1.5, zorder=3.0, alpha=0.5,
            label="Inversion"
        )
ax[2, 1].errorbar(
    lgrid, lgrid*nmt_c, yerr=lgrid*nmt_err, fmt=".", c="C1", lw=1.5, zorder=3.0, alpha=0.5,
    label="NaMaster"
        )
ax[2, 1].errorbar(
    lgrid, lgrid*nu_c, yerr=lgrid*nu_err, fmt=".", c="C2", lw=1.5, zorder=3.0, alpha=0.5,
    label="Natural Unmixing"
        )
ax[2, 1].errorbar(
    lgrid, lgrid*pp_c, yerr=lgrid*pp_err, fmt=".", c="C3", lw=1.5, zorder=3.0, alpha=0.5,
    label="PolSpice"
        )
ax[2, 1].plot(ls[10:], ls[10:]*t[10:], c="k", lw=1.0, zorder=4.0)
ax[3, 1].axhline(0.0, c="k", lw=0.8, zorder=-1)
ax[3, 1].errorbar(
    lgrid,
    (i_c - t_itp) / err,
    yerr=np.abs(i_err / err),
    fmt=".",
    c="C0",
    lw=1.5,
    zorder=1.0,
    alpha=0.5,
)
ax[3, 1].errorbar(
    lgrid,
    (nu_c - t_itp) / err,
    yerr=np.abs(nu_err / err),
    fmt=".",
    c="C1",
    lw=1.5,
    zorder=1.0,
    alpha=0.5,
)
ax[3, 1].errorbar(
    lgrid,
    (pp_c - t_itp) / err,
    yerr=np.abs(pp_err / err),
    fmt=".",
    c="C2",
    lw=1.5,
    zorder=1.0,
    alpha=0.5,
)
ax[2, 1].tick_params(axis="both", which="both", direction="in")
ax[3, 1].axhline(0.0, c="k", lw=0.8, zorder=-1)

ax[2, 1].set_xscale("log")
ax[2, 1].set_xlim(7, lmax * 1.5)
ax[3, 1].set_yscale("symlog", linthresh=1e0)
ax[3, 1].set_ylim(-50, 50)
ax[3, 1].set_xscale("log")
ax[3, 1].set_xlim(7, lmax * 1.5)
ax[3, 1].set_xlabel(r"$\ell$")

# EE
key = ("SHE", "SHE", 1, 1)
cov_key = ("SHE", "SHE", "SHE", "SHE", 1, 1, 1, 1)

cov = ensemble_covqq[cov_key][0, 0, 0, 0, :, :]
i_cov = inv_ensemble_covqq[cov_key][0, 0, 0, 0, :, :]
nmt_cov = nmt_ensemble_covqq[cov_key][0, 0, 0, 0, :, :]
nu_cov = nu_ensemble_covqq[cov_key][0, 0, 0, 0, :, :]
pp_cov = pols_ensemble_covqq[cov_key][0, 0, 0, 0, :, :]

err = np.sqrt(np.diag(cov))/fsky_dr1
i_err = np.sqrt(np.diag(i_cov))
nmt_err = np.sqrt(np.diag(nmt_cov))
nu_err = np.sqrt(np.diag(nu_cov))
pp_err = np.sqrt(np.diag(pp_cov))

i_c = inv_cqs_m[key][0, 0, :]
nmt_c = nmt_cqs_m[key][0, 0, :]
nu_c = nu_cqs_m[key][0, 0, :]
pp_c = pols_cqs_m[key][0, 0, :]
t = _theory_cls[key][0, 0, :]
t_itp = np.interp(lgrid, ls, t)

ax[2, 2].errorbar(
            lgrid, lgrid*i_c, yerr=lgrid*i_err, fmt=".", c="C0", lw=1.5, zorder=3.0, alpha=0.5,
            label="Inversion"
        )
ax[2, 2].errorbar(
    lgrid, lgrid*nmt_c, yerr=lgrid*nmt_err, fmt=".", c="C1", lw=1.5, zorder=3.0, alpha=0.5,
    label="NaMaster"
        )
ax[2, 2].errorbar(
    lgrid, lgrid*nu_c, yerr=lgrid*nu_err, fmt=".", c="C2", lw=1.5, zorder=3.0, alpha=0.5,
    label="Natural Unmixing"
        )
ax[2, 2].errorbar(
    lgrid, lgrid*pp_c, yerr=lgrid*pp_err, fmt=".", c="C3", lw=1.5, zorder=3.0, alpha=0.5,
    label="PolSpice"
        )
ax[2, 2].plot(ls[10:], ls[10:]*t[10:], c="k", lw=1.0, zorder=4.0)
ax[3, 2].axhline(0.0, c="k", lw=0.8, zorder=-1)
ax[3, 2].errorbar(
    lgrid,
    (i_c - t_itp) / err,
    yerr=np.abs(i_err / err),
    fmt=".",
    c="C0",
    lw=1.5,
    zorder=1.0,
    alpha=0.5,
)
ax[3, 2].errorbar(
    lgrid,
    (nu_c - t_itp) / err,
    yerr=np.abs(nu_err / err),
    fmt=".",
    c="C1",
    lw=1.5,
    zorder=1.0,
    alpha=0.5,
)
ax[3, 2].errorbar(
    lgrid,
    (pp_c - t_itp) / err,
    yerr=np.abs(pp_err / err),
    fmt=".",
    c="C2",
    lw=1.5,
    zorder=1.0,
    alpha=0.5,
)
ax[3, 2].axhline(0.0, c="k", lw=0.8, zorder=-1)
ax[2, 2].tick_params(axis="both", which="both", direction="in")
ax[2, 2].set_xscale("log")
ax[2, 2].set_xlim(7, lmax * 1.5)
ax[3, 2].set_yscale("symlog", linthresh=1e0)
ax[3, 2].set_ylim(-50, 50)
ax[3, 2].set_xscale("log")
ax[3, 2].set_xlim(7, lmax * 1.5)
ax[3, 2].set_xlabel(r"$\ell$")

# BB
key = ("SHE", "SHE", 1, 1)
cov_key = ("SHE", "SHE", "SHE", "SHE", 1, 1, 1, 1)

cov = ensemble_covqq[cov_key][1, 1, 1, 1, :, :]
i_cov = inv_ensemble_covqq[cov_key][1, 1, 1, 1, :, :]
nmt_cov = nmt_ensemble_covqq[cov_key][1, 1, 1, 1, :, :]
nu_cov = nu_ensemble_covqq[cov_key][1, 1, 1, 1, :, :]
pp_cov = pols_ensemble_covqq[cov_key][1, 1, 1, 1, :, :]

err = np.sqrt(np.diag(cov))/fsky_dr1
i_err = np.sqrt(np.diag(i_cov))
nmt_err = np.sqrt(np.diag(nmt_cov))
nu_err = np.sqrt(np.diag(nu_cov))
pp_err = np.sqrt(np.diag(pp_cov))

i_c = inv_cqs_m[key][1, 1, :]
nmt_c = nmt_cqs_m[key][1, 1, :]
nu_c = nu_cqs_m[key][1, 1, :]
pp_c = pols_cqs_m[key][1, 1, :]
t = _theory_cls[key][1, 1, :]
t_itp = np.interp(lgrid, ls, t)


ax[2, 3].errorbar(
            lgrid, lgrid*i_c, yerr=lgrid*i_err, fmt=".", c="C0", lw=1.5, zorder=3.0, alpha=0.5,
            label="Inversion"
        )
ax[2, 3].errorbar(
    lgrid, lgrid*nmt_c, yerr=lgrid*nmt_err, fmt=".", c="C1", lw=1.5, zorder=3.0, alpha=0.5,
    label="NaMaster"
        )
ax[2, 3].errorbar(
    lgrid, lgrid*nu_c, yerr=lgrid*nu_err, fmt=".", c="C2", lw=1.5, zorder=3.0, alpha=0.5,
    label="Natural Unmixing"
        )
ax[2, 3].errorbar(
    lgrid, lgrid*pp_c, yerr=lgrid*pp_err, fmt=".", c="C3", lw=1.5, zorder=3.0, alpha=0.5,
    label="PolSpice"
    )
ax[2, 3].plot(ls[10:], ls[10:]*t[10:], c="k", lw=1.0, zorder=4.0)
ax[3, 3].axhline(0.0, c="k", lw=0.8, zorder=-1)
ax[3, 3].errorbar(
    lgrid,
    (i_c - t_itp) / err,
    yerr=np.abs(i_err / err),
    fmt=".",
    c="C0",
    lw=1.5,
    zorder=1.0,
    alpha=0.5,
)
ax[3, 3].errorbar(
    lgrid,
    (nmt_c - t_itp) / err,
    yerr=np.abs(nmt_err / err),
    fmt=".",
    c="C1",
    lw=1.5,
    zorder=1.0,
    alpha=0.5,
)
ax[3, 3].errorbar(
    lgrid,
    (nu_c - t_itp) / err,
    yerr=np.abs(nu_err / err),
    fmt=".",
    c="C2",
    lw=1.5,
    zorder=1.0,
    alpha=0.5,
)
ax[3, 3].errorbar(
    lgrid,
    (pp_c - t_itp) / err,
    yerr=np.abs(pp_err / err),
    fmt=".",
    c="C3", 
    lw=1.5,
    zorder=1.0,
    alpha=0.5,
)

ax[2, 3].legend()
ax[2, 3].tick_params(axis="both", which="both", direction="in")
ax[3, 3].axhline(0.0, c="k", lw=0.8, zorder=-1)
ax[2, 3].set_xscale("log")
ax[2, 3].set_xlim(7, lmax * 1.5)
ax[3, 3].set_yscale("symlog", linthresh=1e0)
ax[3, 3].set_ylim(-50, 50)
ax[3, 3].set_xscale("log")
ax[3, 3].set_xlim(7, lmax * 1.5)
ax[3, 3].set_xlabel(r"$\ell$")

#######
## patch
#######

ensemble_covqq = dr1_ensemble_covqq
inv_ensemble_covqq = dr1_inv_ensemble_covqq
nmt_ensemble_covqq = dr1_nmt_ensemble_covqq
nu_ensemble_covqq = dr1_nu_ensemble_covqq
pols_ensemble_covqq = dr1_pols_ensemble_covqq

inv_cqs_m = dr1_inv_cqs_m
nmt_cqs_m = dr1_nmt_cqs_m
nu_cqs_m = dr1_nu_cqs_m
pols_cqs_m = dr1_pols_cqs_m

# POS
key = ("POS", "POS", 1, 1)
cov_key = ("POS", "POS", "POS", "POS", 1, 1, 1, 1)

cov = ensemble_covqq[cov_key]
i_cov = inv_ensemble_covqq[cov_key]
nmt_cov = nmt_ensemble_covqq[cov_key]
nu_cov = nu_ensemble_covqq[cov_key]
pp_cov = pols_ensemble_covqq[cov_key]

err = np.sqrt(np.diag(cov))/fsky_patch
i_err = np.sqrt(np.diag(i_cov))
nmt_err = np.sqrt(np.diag(nmt_cov))
nu_err = np.sqrt(np.diag(nu_cov))
pp_err = np.sqrt(np.diag(pp_cov))

i_c = inv_cqs_m[key]
nmt_c = nmt_cqs_m[key]
nu_c = nu_cqs_m[key]
pp_c = pols_cqs_m[key]
t = _theory_cls[key]
t_itp = np.interp(lgrid, ls, t)


ax[4, 0].errorbar(
            lgrid, lgrid*i_c, yerr=lgrid*i_err, fmt=".", c="C0", lw=1.5, zorder=3.0, alpha=0.5,
            label="Inversion"
        )
ax[4, 0].errorbar(
    lgrid, lgrid*nmt_c, yerr=lgrid*nmt_err, fmt=".", c="C1", lw=1.5, zorder=3.0, alpha=0.5,
    label="NaMaster"
        )
ax[4, 0].errorbar(
    lgrid, lgrid*nu_c, yerr=lgrid*nu_err, fmt=".", c="C2", lw=1.5, zorder=3.0, alpha=0.5,
    label="Natural Unmixing"
        )
ax[4, 0].errorbar(
    lgrid, lgrid*pp_c, yerr=lgrid*pp_err, fmt=".", c="C3", lw=1.5, zorder=3.0, alpha=0.5,
    label="PolSpice"
    )
ax[4, 0].plot(ls[10:], ls[10:]*t[10:], c="k", lw=1.0, zorder=4.0)
ax[5, 0].axhline(0.0, c="k", lw=0.8, zorder=-1)
ax[5, 0].errorbar(
    lgrid,
    (i_c - t_itp) / err,
    yerr=np.abs(i_err / err),
    fmt=".",
    c="C0",
    lw=1.5,
    zorder=1.0,
    alpha=0.5,
)
ax[5, 0].errorbar(
    lgrid,
    (nmt_c - t_itp) / err,
    yerr=np.abs(nmt_err / err),
    fmt=".",
    c="C1",
    lw=1.5,
    zorder=1.0,
    alpha=0.5,
)
ax[5, 0].errorbar(
    lgrid,
    (nu_c - t_itp) / err,
    yerr=np.abs(nu_err / err),
    fmt=".",
    c="C2",
    lw=1.5,
    zorder=1.0,
    alpha=0.5,
)
ax[5, 0].errorbar(
    lgrid,
    (pp_c - t_itp) / err,
    yerr=np.abs(pp_err / err),
    fmt=".",
    c="C3", 
    lw=1.5,
    zorder=1.0,
    alpha=0.5,
)
ax[5, 0].axhline(0.0, c="k", lw=0.8, zorder=-1)

ax[4, 0].tick_params(axis="both", which="both", direction="in")

ax[4, 0].set_xscale("log")
ax[4, 0].set_xlim(7, lmax * 1.5)
ax[4, 0].set_ylabel("DR1 - $\ell C_\ell$")
ax[5, 0].set_yscale("symlog", linthresh=1e0)
ax[5, 0].set_ylim(-50, 50)
ax[5, 0].set_xscale("log")
ax[5, 0].set_xlim(7, lmax * 1.5)
ax[5, 0].set_xlabel("$\ell$")
ax[5, 0].set_ylabel(r"$\frac{\ell \Delta C_\ell}{\sigma_{C_\ell}}$")

# POS-E
key = ("POS", "SHE", 1, 1)
cov_key = ("POS", "SHE", "POS", "SHE", 1, 1, 1, 1)

cov = ensemble_covqq[cov_key][0, 0, :, :]
i_cov = inv_ensemble_covqq[cov_key][0, 0, :, :]
nmt_cov = nmt_ensemble_covqq[cov_key][0, 0, :, :]
nu_cov = nu_ensemble_covqq[cov_key][0, 0, :, :]
pp_cov = pols_ensemble_covqq[cov_key][0, 0, :, :]

err = np.sqrt(np.diag(cov))/fsky_patch
i_err = np.sqrt(np.diag(i_cov))
nmt_err = np.sqrt(np.diag(nmt_cov))
nu_err = np.sqrt(np.diag(nu_cov))
pp_err = np.sqrt(np.diag(pp_cov))

i_c = inv_cqs_m[key][0, :]
nmt_c = nmt_cqs_m[key][0, :]
nu_c = nu_cqs_m[key][0, :]
pp_c = pols_cqs_m[key][0, :]
t = _theory_cls[key][0, :]
t_itp = np.interp(lgrid, ls, t)

ax[4, 1].errorbar(
            lgrid, lgrid*i_c, yerr=lgrid*i_err, fmt=".", c="C0", lw=1.5, zorder=3.0, alpha=0.5,
            label="Inversion"
        )
ax[4, 1].errorbar(
    lgrid, lgrid*nmt_c, yerr=lgrid*nmt_err, fmt=".", c="C1", lw=1.5, zorder=3.0, alpha=0.5,
    label="NaMaster"
        )
ax[4, 1].errorbar(
    lgrid, lgrid*nu_c, yerr=lgrid*nu_err, fmt=".", c="C2", lw=1.5, zorder=3.0, alpha=0.5,
    label="Natural Unmixing"
        )
ax[4, 1].errorbar(
    lgrid, lgrid*pp_c, yerr=lgrid*pp_err, fmt=".", c="C3", lw=1.5, zorder=3.0, alpha=0.5,
    label="PolSpice"
        )
ax[4, 1].plot(ls[10:], ls[10:]*t[10:], c="k", lw=1.0, zorder=4.0)
ax[5, 1].axhline(0.0, c="k", lw=0.8, zorder=-1)
ax[5, 1].errorbar(
    lgrid,
    (i_c - t_itp) / err,
    yerr=np.abs(i_err / err),
    fmt=".",
    c="C0",
    lw=1.5,
    zorder=1.0,
    alpha=0.5,
)
ax[5, 1].errorbar(
    lgrid,
    (nu_c - t_itp) / err,
    yerr=np.abs(nu_err / err),
    fmt=".",
    c="C1",
    lw=1.5,
    zorder=1.0,
    alpha=0.5,
)
ax[5, 1].errorbar(
    lgrid,
    (pp_c - t_itp) / err,
    yerr=np.abs(pp_err / err),
    fmt=".",
    c="C2",
    lw=1.5,
    zorder=1.0,
    alpha=0.5,
)
ax[4, 1].tick_params(axis="both", which="both", direction="in")

ax[4, 1].set_xscale("log")
ax[4, 1].set_xlim(7, lmax * 1.5)
ax[5, 1].set_yscale("symlog", linthresh=1e0)
ax[5, 1].set_ylim(-50, 50)
ax[5, 1].set_xscale("log")
ax[5, 1].set_xlim(7, lmax * 1.5)
ax[5, 1].set_xlabel(r"$\ell$")

# EE
key = ("SHE", "SHE", 1, 1)
cov_key = ("SHE", "SHE", "SHE", "SHE", 1, 1, 1, 1)

cov = ensemble_covqq[cov_key][0, 0, 0, 0, :, :]
i_cov = inv_ensemble_covqq[cov_key][0, 0, 0, 0, :, :]
nmt_cov = nmt_ensemble_covqq[cov_key][0, 0, 0, 0, :, :]
nu_cov = nu_ensemble_covqq[cov_key][0, 0, 0, 0, :, :]
pp_cov = pols_ensemble_covqq[cov_key][0, 0, 0, 0, :, :]

err = np.sqrt(np.diag(cov))/fsky_patch
i_err = np.sqrt(np.diag(i_cov))
nmt_err = np.sqrt(np.diag(nmt_cov))
nu_err = np.sqrt(np.diag(nu_cov))
pp_err = np.sqrt(np.diag(pp_cov))

i_c = inv_cqs_m[key][0, 0, :]
nmt_c = nmt_cqs_m[key][0, 0, :]
nu_c = nu_cqs_m[key][0, 0, :]
pp_c = pols_cqs_m[key][0, 0, :]
t = _theory_cls[key][0, 0, :]
t_itp = np.interp(lgrid, ls, t)

ax[4, 2].errorbar(
            lgrid, lgrid*i_c, yerr=lgrid*i_err, fmt=".", c="C0", lw=1.5, zorder=3.0, alpha=0.5,
            label="Inversion"
        )
ax[4, 2].errorbar(
    lgrid, lgrid*nmt_c, yerr=lgrid*nmt_err, fmt=".", c="C1", lw=1.5, zorder=3.0, alpha=0.5,
    label="NaMaster"
        )
ax[4, 2].errorbar(
    lgrid, lgrid*nu_c, yerr=lgrid*nu_err, fmt=".", c="C2", lw=1.5, zorder=3.0, alpha=0.5,
    label="Natural Unmixing"
        )
ax[4, 2].errorbar(
    lgrid, lgrid*pp_c, yerr=lgrid*pp_err, fmt=".", c="C3", lw=1.5, zorder=3.0, alpha=0.5,
    label="PolSpice"
        )
ax[4, 2].plot(ls[10:], ls[10:]*t[10:], c="k", lw=1.0, zorder=4.0)
ax[5, 2].axhline(0.0, c="k", lw=0.8, zorder=-1)
ax[5, 2].errorbar(
    lgrid,
    (i_c - t_itp) / err,
    yerr=np.abs(i_err / err),
    fmt=".",
    c="C0",
    lw=1.5,
    zorder=1.0,
    alpha=0.5,
)
ax[5, 2].errorbar(
    lgrid,
    (nu_c - t_itp) / err,
    yerr=np.abs(nu_err / err),
    fmt=".",
    c="C1",
    lw=1.5,
    zorder=1.0,
    alpha=0.5,
)
ax[5, 2].errorbar(
    lgrid,
    (pp_c - t_itp) / err,
    yerr=np.abs(pp_err / err),
    fmt=".",
    c="C2",
    lw=1.5,
    zorder=1.0,
    alpha=0.5,
)
ax[5, 2].tick_params(axis="both", which="both", direction="in")
ax[5, 2].set_xscale("log")
ax[5, 2].set_xlim(7, lmax * 1.5)
ax[5, 2].set_yscale("symlog", linthresh=1e0)
ax[5, 2].set_ylim(-50, 50)
ax[4, 2].set_xscale("log")
ax[4, 2].set_xlim(7, lmax * 1.5)
ax[4, 2].set_xlabel(r"$\ell$")

# BB
key = ("SHE", "SHE", 1, 1)
cov_key = ("SHE", "SHE", "SHE", "SHE", 1, 1, 1, 1)

cov = ensemble_covqq[cov_key][1, 1, 1, 1, :, :]
i_cov = inv_ensemble_covqq[cov_key][1, 1, 1, 1, :, :]
nmt_cov = nmt_ensemble_covqq[cov_key][1, 1, 1, 1, :, :]
nu_cov = nu_ensemble_covqq[cov_key][1, 1, 1, 1, :, :]
pp_cov = pols_ensemble_covqq[cov_key][1, 1, 1, 1, :, :]

err = np.sqrt(np.diag(cov))/fsky_patch
i_err = np.sqrt(np.diag(i_cov))
nmt_err = np.sqrt(np.diag(nmt_cov))
nu_err = np.sqrt(np.diag(nu_cov))
pp_err = np.sqrt(np.diag(pp_cov))

i_c = inv_cqs_m[key][1, 1, :]
nmt_c = nmt_cqs_m[key][1, 1, :]
nu_c = nu_cqs_m[key][1, 1, :]
pp_c = pols_cqs_m[key][1, 1, :]
t = _theory_cls[key][1, 1, :]
t_itp = np.interp(lgrid, ls, t)


ax[4, 3].errorbar(
            lgrid, lgrid*i_c, yerr=lgrid*i_err, fmt=".", c="C0", lw=1.5, zorder=3.0, alpha=0.5,
            label="Inversion"
        )
ax[4, 3].errorbar(
    lgrid, lgrid*nmt_c, yerr=lgrid*nmt_err, fmt=".", c="C1", lw=1.5, zorder=3.0, alpha=0.5,
    label="NaMaster"
        )
ax[4, 3].errorbar(
    lgrid, lgrid*nu_c, yerr=lgrid*nu_err, fmt=".", c="C2", lw=1.5, zorder=3.0, alpha=0.5,
    label="Natural Unmixing"
        )
ax[4, 3].errorbar(
    lgrid, lgrid*pp_c, yerr=lgrid*pp_err, fmt=".", c="C3", lw=1.5, zorder=3.0, alpha=0.5,
    label="PolSpice"
    )
ax[4, 3].plot(ls[10:], ls[10:]*t[10:], c="k", lw=1.0, zorder=4.0)
ax[5, 3].axhline(0.0, c="k", lw=0.8, zorder=-1)
ax[5, 3].errorbar(
    lgrid,
    (i_c - t_itp) / err,
    yerr=np.abs(i_err / err),
    fmt=".",
    c="C0",
    lw=1.5,
    zorder=1.0,
    alpha=0.5,
)
ax[5, 3].errorbar(
    lgrid,
    (nmt_c - t_itp) / err,
    yerr=np.abs(nmt_err / err),
    fmt=".",
    c="C1",
    lw=1.5,
    zorder=1.0,
    alpha=0.5,
)
ax[5, 3].errorbar(
    lgrid,
    (nu_c - t_itp) / err,
    yerr=np.abs(nu_err / err),
    fmt=".",
    c="C2",
    lw=1.5,
    zorder=1.0,
    alpha=0.5,
)
ax[5, 3].errorbar(
    lgrid,
    (pp_c - t_itp) / err,
    yerr=np.abs(pp_err / err),
    fmt=".",
    c="C3", 
    lw=1.5,
    zorder=1.0,
    alpha=0.5,
)

ax[4, 3].legend()
ax[4, 3].tick_params(axis="both", which="both", direction="in")
ax[5, 3].axhline(0.0, c="k", lw=0.8, zorder=-1)
ax[4, 3].set_xscale("log")
ax[4, 3].set_xlim(7, lmax * 1.5)
ax[5, 3].set_yscale("symlog", linthresh=1e0)
ax[5, 3].set_ylim(-50, 50)
ax[5, 3].set_xscale("log")
ax[5, 3].set_xlim(7, lmax * 1.5)
ax[5, 3].set_xlabel(r"$\ell$")

fig.savefig(f'/home/jaimerzp/Desktop/mixing_mat_plots/cls_comp.pdf', bbox_inches='tight')

: 

: 

In [ ]:
def get_xi2s(cls_data, cls_theory, covs):
    """Calculate the chi-squared value."""
    xi2s = {}
    for key in list(cls_data.keys()):
        a, b, i, j = key
        covkey = (a, b, a, b, i, j, i, j)
        cov = covs[covkey]
        cl_data = cls_data[key]
        cl_theory = cls_theory[key]
        if a == b == "POS":
            cl_data = cl_data.array
            cl_theory = cl_theory.array
            diff = cl_data - cl_theory
            cov = cov.array
            invcov = np.linalg.pinv(cov)
            xi2 = np.dot(diff, np.dot(invcov, diff))
            xi2s[key] = xi2/len(cl_data)
        elif a == b == "SHE":
            cl_data_ee = cl_data[0, 0, :]
            cl_data_eb = cl_data[0, 1, :]
            cl_data_bb = cl_data[1, 1, :]
            cl_theory_ee = cl_theory[0, 0, :]
            cl_theory_eb = cl_theory[0, 1, :]
            cl_theory_bb = cl_theory[1, 1, :]
            diff_ee = cl_data_ee - cl_theory_ee
            diff_eb = cl_data_eb - cl_theory_eb
            diff_bb = cl_data_bb - cl_theory_bb
            cov_ee = cov[0, 0, 0, 0, :, :]
            cov_eb = cov[0, 1, 0, 1, :, :]
            cov_bb = cov[1, 1, 1, 1, :, :]
            invcov_ee = np.linalg.pinv(cov_ee)
            invcov_eb = np.linalg.pinv(cov_eb)
            invcov_bb = np.linalg.pinv(cov_bb)
            xi2_ee = np.dot(diff_ee, np.dot(invcov_ee, diff_ee))
            xi2_eb = np.dot(diff_eb, np.dot(invcov_eb, diff_eb))
            xi2_bb = np.dot(diff_bb, np.dot(invcov_bb, diff_bb))
            xi2s[('E', 'E', i, j)] = xi2_ee/len(cl_data_ee)
            xi2s[('E', 'B', i, j)] = xi2_eb/len(cl_data_eb)
            xi2s[('B', 'B', i, j)] = xi2_bb/len(cl_data_bb)
        elif a == "POS" and b == "SHE":
            cl_pe = cl_data[0, :]
            cl_pb = cl_data[1, :]
            cl_theory_pe = cl_theory[0, :]
            cl_theory_pb = cl_theory[1, :]
            diff_pe = (cl_pe - cl_theory_pe)
            diff_pb = (cl_pb - cl_theory_pb)
            cov_pe = cov[0, 0, :, :]
            cov_pb = cov[1, 1, :, :]
            invcov_pe = np.linalg.pinv(cov_pe)
            invcov_pb = np.linalg.pinv(cov_pb)
            xi2_pe = np.dot(diff_pe, np.dot(invcov_pe, diff_pe))
            xi2_pb = np.dot(diff_pb, np.dot(invcov_pb, diff_pb))
            xi2s[('POS', 'E', i, j)] = xi2_pe/len(cl_pe)
            xi2s[('POS', 'B', i, j)] = xi2_pb/len(cl_pb)
        else:
            raise ValueError(f"Unknown key: {key}")
    return xi2s

: 

: 

In [ ]:
rr2_xi2s_inv = get_xi2s(rr2_inv_cqs_m, _theory_cqs, rr2_inv_ensemble_covqq)
rr2_xi2s_nmt = get_xi2s(rr2_nmt_cqs_m, _theory_cqs, rr2_nmt_ensemble_covqq)
rr2_xi2s_nu = get_xi2s(rr2_nu_cqs_m, _theory_cqs, rr2_nu_ensemble_covqq)
rr2_xi2s_pols = get_xi2s(rr2_pols_cqs_m, _theory_cqs, rr2_pols_ensemble_covqq)

dr1_xi2s_inv = get_xi2s(dr1_inv_cqs_m, _theory_cqs, dr1_inv_ensemble_covqq)
dr1_xi2s_nmt = get_xi2s(dr1_nmt_cqs_m, _theory_cqs, dr1_nmt_ensemble_covqq)
dr1_xi2s_nu = get_xi2s(dr1_nu_cqs_m, _theory_cqs, dr1_nu_ensemble_covqq)
dr1_xi2s_pols = get_xi2s(dr1_pols_cqs_m, _theory_cqs, dr1_pols_ensemble_covqq)

patch_xi2s_inv = get_xi2s(patch_inv_cqs_m, _theory_cqs, patch_inv_ensemble_covqq)
patch_xi2s_nmt = get_xi2s(patch_nmt_cqs_m, _theory_cqs, patch_nmt_ensemble_covqq)
patch_xi2s_nu = get_xi2s(patch_nu_cqs_m, _theory_cqs, patch_nu_ensemble_covqq)
patch_xi2s_pols = get_xi2s(patch_pols_cqs_m, _theory_cqs, patch_pols_ensemble_covqq)

: 

: 

In [ ]:
import matplotlib.pyplot as plt

fig, axes = plt.subplots(3, 1, figsize=(6, 10))
fig.subplots_adjust(hspace=0.0)

###########
## half-sky
###########
chi2_dicts = {
    "Inverse": rr2_xi2s_inv,
    "NaMaster": rr2_xi2s_nmt,
    "Natural Unmixing": rr2_xi2s_nu,
    "PolSpice": rr2_xi2s_pols,
}
keys = list(chi2_dicts.values())[0].keys()
labels = [
    "POSxPOS", "POSxE", "POSxB",
    "ExE", "ExB", "BxB"
]

x = range(len(keys))
width = 0.13
for i, (name, d) in enumerate(chi2_dicts.items()):
    values = [d[k]*fsky_rr2**2 for k in keys]
    axes[0].bar(
        [xi + i * width for xi in x],
        values,
        width=width,
        label=name
    )

axes[0].set_xticks([])
axes[0].set_xticklabels([])
axes[0].set_ylabel(r"$\chi^2$")
axes[0].set_title("RR2 Mask", y=0.9)
axes[0].set_yscale("log")

#########
## Planck
#########
chi2_dicts = {
    "Inverse": patch_xi2s_inv,
    "NaMaster": patch_xi2s_nmt,
    "Natural Unmixing": patch_xi2s_nu,
    "PolSpice": patch_xi2s_pols,
}
keys = list(chi2_dicts.values())[0].keys()
labels = [
    "POSxPOS", "POSxE", "POSxB",
    "ExE", "ExB", "BxB"
]

x = range(len(keys))
width = 0.13
for i, (name, d) in enumerate(chi2_dicts.items()):
    values = [d[k]*fsky_dr1**2 for k in keys]
    axes[1].bar(
        [xi + i * width for xi in x],
        values,
        width=width,
        label=name
    )

axes[1].set_xticks([])
axes[1].set_xticklabels([])
axes[1].set_ylabel(r"$\chi^2$")
axes[1].set_title("DR1 South Mask", y=0.9)
axes[1].set_yscale("log")

########
## patch
########
chi2_dicts = {
    "Inverse": dr1_xi2s_inv,
    "NaMaster": dr1_xi2s_nmt,
    "Natural Unmixing": dr1_xi2s_nu,
    "PolSpice": dr1_xi2s_pols,
}
keys = list(chi2_dicts.values())[0].keys()
labels = [
    "POSxPOS", "POSxE", "POSxB",
    "ExE", "ExB", "BxB"
]

x = range(len(keys))
width = 0.13
for i, (name, d) in enumerate(chi2_dicts.items()):
    values = [d[k]*fsky_patch**2 for k in keys]
    axes[2].bar(
        [xi + i * width for xi in x],
        values,
        width=width,
        label=name
    )

axes[2].set_xticks([xi + width*2.5 for xi in x])
axes[2].set_xticklabels(labels)
axes[2].set_ylabel(r"$\chi^2$")
axes[2].set_title("DR1 Mask", y=0.9)
axes[2].set_yscale("log")
axes[2].legend()

plt.tight_layout()
plt.show()
fig.savefig(f'/home/jaimerzp/Desktop/mixing_mat_plots/Xi2s_comp.pdf', bbox_inches='tight')

: 

: 

In [ ]:
rr2_xi2s_inv = get_xi2s(rr2_inv_cqs_m, _theory_cqs, rr2_ensemble_covqq)
rr2_xi2s_nmt = get_xi2s(rr2_nmt_cqs_m, _theory_cqs, rr2_ensemble_covqq)
rr2_xi2s_nu = get_xi2s(rr2_nu_cqs_m, _theory_cqs, rr2_ensemble_covqq)
rr2_xi2s_pols = get_xi2s(rr2_pols_cqs_m, _theory_cqs, rr2_ensemble_covqq)

patch_xi2s_inv = get_xi2s(patch_inv_cqs_m, _theory_cqs, patch_ensemble_covqq)
patch_xi2s_nmt = get_xi2s(patch_nmt_cqs_m, _theory_cqs, patch_ensemble_covqq)
patch_xi2s_nu = get_xi2s(patch_nu_cqs_m, _theory_cqs, patch_ensemble_covqq)
patch_xi2s_pols = get_xi2s(patch_pols_cqs_m, _theory_cqs, patch_ensemble_covqq)

dr1_xi2s_inv = get_xi2s(dr1_inv_cqs_m, _theory_cqs, dr1_ensemble_covqq)
dr1_xi2s_nmt = get_xi2s(dr1_nmt_cqs_m, _theory_cqs, dr1_ensemble_covqq)
dr1_xi2s_nu = get_xi2s(dr1_nu_cqs_m, _theory_cqs, dr1_ensemble_covqq)
dr1_xi2s_pols = get_xi2s(dr1_pols_cqs_m, _theory_cqs, dr1_ensemble_covqq)

: 

: 

In [ ]:
import matplotlib.pyplot as plt

fig, axes = plt.subplots(3, 1, figsize=(6, 10))
fig.subplots_adjust(hspace=0.0)

###########
## half-sky
###########
chi2_dicts = {
    "Inverse": rr2_xi2s_inv,
    "NaMaster": rr2_xi2s_nmt,
    "Natural Unmixing": rr2_xi2s_nu,
    "PolSpice": rr2_xi2s_pols,
}
keys = list(chi2_dicts.values())[0].keys()
labels = [
    "POSxPOS", "POSxE", "POSxB",
    "ExE", "ExB", "BxB"
]

x = range(len(keys))
width = 0.13
for i, (name, d) in enumerate(chi2_dicts.items()):
    values = [d[k]*fsky_rr2**2 for k in keys]
    axes[0].bar(
        [xi + i * width for xi in x],
        values,
        width=width,
        label=name
    )

axes[0].set_xticks([])
axes[0].set_xticklabels([])
axes[0].set_ylabel(r"$\chi^2$")
axes[0].set_title("RR2 Mask", y=0.9)
axes[0].set_yscale("log")

#########
## Planck
#########
chi2_dicts = {
    "Inverse": patch_xi2s_inv,
    "NaMaster": patch_xi2s_nmt,
    "Natural Unmixing": patch_xi2s_nu,
    "PolSpice": patch_xi2s_pols,
}
keys = list(chi2_dicts.values())[0].keys()
labels = [
    "POSxPOS", "POSxE", "POSxB",
    "ExE", "ExB", "BxB"
]

x = range(len(keys))
width = 0.13
for i, (name, d) in enumerate(chi2_dicts.items()):
    values = [d[k]*fsky_dr1**2 for k in keys]
    axes[1].bar(
        [xi + i * width for xi in x],
        values,
        width=width,
        label=name
    )

axes[1].set_xticks([])
axes[1].set_xticklabels([])
axes[1].set_ylabel(r"$\chi^2$")
axes[1].set_title("DR1 South Mask", y=0.9)
axes[1].set_yscale("log")

########
## patch
########
chi2_dicts = {
    "Inverse": dr1_xi2s_inv,
    "NaMaster": dr1_xi2s_nmt,
    "Natural Unmixing": dr1_xi2s_nu,
    "PolSpice": dr1_xi2s_pols,
}
keys = list(chi2_dicts.values())[0].keys()
labels = [
    "POSxPOS", "POSxE", "POSxB",
    "ExE", "ExB", "BxB"
]

x = range(len(keys))
width = 0.13
for i, (name, d) in enumerate(chi2_dicts.items()):
    values = [d[k]*fsky_patch**2 for k in keys]
    axes[2].bar(
        [xi + i * width for xi in x],
        values,
        width=width,
        label=name
    )

axes[2].set_xticks([xi + width*2.5 for xi in x])
axes[2].set_xticklabels(labels)
axes[2].set_ylabel(r"$\chi^2$")
axes[2].set_title("DR1 Mask", y=0.9)
axes[2].set_yscale("log")
axes[2].legend()

plt.tight_layout()
plt.show()
fig.savefig(f'/home/jaimerzp/Desktop/mixing_mat_plots/Xi2s_comp.pdf', bbox_inches='tight')

: 

: 

: 

: 

: 

: 